# Linguistic analysis of `claude-code-system-prompts`

This notebook profiles the 287 prompt files at
`claude-prompts-analysis/claude-code-system-prompts/system-prompts/` along eight dimensions:

1. **Mood** — grammatical mood via spaCy parse + `IMPERATIVE_MARKERS` lexical match.
2. **Register** — TTR, sentence length, dependency depth, F-score, plus four-class lexical register
   (`frozen` / `formal` / `consultative` / `casual`).
3. **Stance** — four-class lexical stance (`directive` / `expository` / `evaluative` / `dialogic`),
   plus 1st/2nd-person engagement.
4. **Modality** — single spaCy parse-tree detector classifying every modal expression as
   **deontic** / **epistemic** / **dynamic**. Catches single-token modals (`should`, `must`),
   multi-word constructions (`should be`, `have to`, `is able to`), and adverbial epistemics
   (`likely`, `probably`).
5. **Vocabulary profile** — full `VOCAB` scan (prohibitions, prescriptions, politeness, warmth,
   hedging, structural, profanity, pronouns).
6. **ALL CAPS emphasis** — uppercase tokens with `TECH_ACRONYMS` excluded.
7. **CAPS imperative tokens** — direct match on `CAPS_IMPERATIVE_TOKENS`.
8. **Justification patterns** — `JUSTIFICATION_PATTERNS` regex match + justification ratio.

**Units — every density metric reports two parallel views:**
- `pct`: matches as a **percentage of word tokens** (`100 × matches / n_tokens`).
- `per_sent`: matches as **average rate per sentence** (`matches / n_sents`).

Sentence-level mood ratios use the `_sent_pct` suffix (% of sentences classified as imperative /
declarative / interrogative).

**Field-naming contract** inside every metric block: `count` (raw integer), `pct` (% of words),
`per_sent` (avg per sentence), plus block-specific extras (`top`, `hits`, `ratio`, `sent_pct`,
domain-specific keys for VOCAB / register / stance classes). No block-name prefixes leak into
inner field names.

Every metric is computed at two levels: **per-file** (one entry per `.md` file) and
**corpus-wide** (aggregate across all 287 files, plus per-category breakdown).

All results are written to a single YAML file:
**`claude-prompts-analysis/prompt_linguistic_analysis.yaml`**.

In [1]:
%pip install --quiet spacy python-frontmatter
!python -m spacy download en_core_web_sm --quiet


Note: you may need to restart the kernel to use updated packages.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
"""Path configuration. Notebook lives at the project root (claude-prompts-analysis/).
The kernel CWD is the same directory, so all paths are relative to it."""
import os, pathlib

# Move kernel CWD into the project root if started elsewhere
PROJECT_ROOT = pathlib.Path("/home/user/workspace/claude-prompts-analysis").resolve()
if pathlib.Path.cwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

CORPUS_DIR = PROJECT_ROOT / "claude-code-system-prompts" / "system-prompts"
YAML_OUT   = PROJECT_ROOT / "prompt_linguistic_analysis.yaml"

assert CORPUS_DIR.is_dir(), f"missing corpus dir: {CORPUS_DIR}"
md_files = sorted(CORPUS_DIR.glob("*.md"))
print(f"cwd:    {pathlib.Path.cwd()}")
print(f"corpus: {CORPUS_DIR}")
print(f"        {len(md_files)} .md files")
print(f"output: {YAML_OUT}")


cwd:    /home/user/workspace/claude-prompts-analysis
corpus: /home/user/workspace/claude-prompts-analysis/claude-code-system-prompts/system-prompts
        287 .md files
output: /home/user/workspace/claude-prompts-analysis/prompt_linguistic_analysis.yaml


In [3]:
"""Imports + spaCy pipeline."""
import re
import datetime as dt
from collections import Counter, defaultdict
from typing import Dict, List, Tuple

import frontmatter
import pandas as pd
import spacy
import yaml
from spacy.matcher import PhraseMatcher
from tqdm.auto import tqdm

NLP = spacy.load("en_core_web_sm", disable=["ner"])
print("spaCy", spacy.__version__, "model", NLP.meta["name"], NLP.meta["version"])
print("pipeline:", NLP.pipe_names)


spaCy 3.8.14 model core_web_sm 3.8.0
pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer']


In [4]:
"""Lexicon definitions — single source of truth.

All dictionaries below are echoed verbatim into the YAML output for reproducibility.
"""

VOCAB: Dict[str, List[str]] = {
    "hard_prohibitions": [
        "do not", "don't", "must not", "mustn't", "never", "cannot", "can't",
        "won't", "shall not", "shouldn't", "should not", "forbidden",
        "prohibited", "disallowed", "not allowed", "not permitted",
        "under no circumstances", "no exceptions", "refuse to",
    ],
    "hard_prescriptions": [
        "must", "always", "required", "mandatory", "critical", "essential",
        "imperative", "obligatory", "you must", "you have to", "you need to",
        "you are required", "you should always", "is required",
    ],
    "soft_prescriptions": [
        "should", "ought to", "make sure", "be sure to", "ensure",
        "remember to", "be careful", "take care to", "you should",
        "try to", "aim to",
    ],
    "politeness_direct": [
        "please", "kindly", "thank you", "thanks", "thank",
        "appreciate", "appreciated", "grateful",
        "sorry", "apologies", "apologize", "apologise", "pardon",
    ],
    "politeness_softening": [
        "if you'd like", "if you would like", "if you want",
        "feel free", "feel free to", "no rush", "take your time",
        "would you mind", "would you", "could you", "would it be possible",
        "if possible", "perhaps", "maybe you could", "if you can",
    ],
    "warmth_encouragement": [
        "welcome", "you're welcome", "glad", "happy to", "happy that",
        "hope", "hopefully", "well done", "good job", "great job",
        "nice work", "great work", "wonderful", "excellent", "amazing",
        "love", "enjoy", "lovely", "fantastic", "brilliant", "delighted",
        "proud", "congratulations", "congrats",
    ],
    "hedging": [
        "may", "might", "could", "possibly", "potentially", "likely",
        "probably", "perhaps", "sometimes", "often", "usually",
        "generally", "typically", "tends to", "in some cases",
    ],
    "structural_markers": [
        "note", "warning", "caution", "important", "tip", "info",
        "remember", "example", "for instance", "for example",
    ],
    "profanity": [
        "fuck", "fucking", "fucked", "fucks",
        "shit", "shitty",
        "damn", "damned", "damnit", "dammit", "hell",
        "ass", "asshole", "bullshit", "crap", "crappy", "bitch",
    ],
    "pronouns_2p": ["you", "your", "yours", "yourself"],
    "pronouns_1p": ["i", "me", "my", "we", "us", "our"],
}

# Authoritative imperative-style markers used to compute mood.marker_*.
# Audit-extended with phrases attested >=5 times in the 287-file corpus.
IMPERATIVE_MARKERS = [
    "must", "must not", "mustn't", "never", "always",
    "do not", "don't", "cannot", "can't", "won't",
    "should not", "shouldn't", "required", "mandatory", "critical",
    "forbidden", "prohibited", "disallowed",
    "you must", "you must not", "you have to", "you need to",
    "you should always",
    "is required", "no exceptions", "under no circumstances",
    # audit additions
    "ensure", "ensure that", "make sure to", "be sure to",
]

CAPS_IMPERATIVE_TOKENS = [
    "IMPORTANT", "VERY IMPORTANT", "CRITICAL", "MANDATORY", "REQUIRED",
    "MUST", "MUST NOT", "NEVER", "ALWAYS", "DO NOT", "DON'T",
    "SHOULD", "SHOULD NOT", "WARNING", "CAUTION", "NOTE",
    "STOP", "ATTENTION", "REMEMBER", "FORBIDDEN", "PROHIBITED",
    "ABSOLUTELY", "STRICTLY",
]

JUSTIFICATION_PATTERNS = [
    r"\bbecause\b", r"\bbecause of\b", r"\bdue to\b",
    r"\bthe reason (?:is|being|that|why)\b",
    r"\b(?:that's|that is) why\b",
    r"\bin order to\b", r"\bso (?:that|as to)\b",
    r"\bso (?:we|you|it|they|the|this|i)\b",
    r"\bto avoid\b", r"\bto prevent\b", r"\bto ensure\b",
    r"\bto guarantee\b", r"\bto make sure\b",
    r"\bthe goal (?:is|of)\b", r"\bthe purpose (?:is|of)\b",
    r"\bthe point (?:is|of)\b",
    r"\botherwise\b", r"\bor else\b", r"\bif not\b",
    r"\bif you (?:don't|do not)\b",
    r"\b(?:could|may|might|would|will) (?:cause|result|lead|break|fail|"
    r"corrupt|hang|crash|loop|deadlock|delete)\b",
    r"\bleads? to\b", r"\bresults? in\b",
    r"\b(?:risks?|risking)\b",
    r"\bthis (?:ensures|prevents|avoids|allows|helps|means|guarantees|"
    r"is because|is to|lets|gives|keeps|protects|stops)\b",
    r"\bthat way\b", r"\bensures? that\b",
    r"\bprevents? .{1,40}? from\b",
    r"\bfor (?:safety|security|consistency|clarity|reproducibility|"
    r"reliability|correctness|accuracy|performance|the user)\b",
    r"\bsince\b", r"\bgiven that\b", r"\bas a result\b",
]

TECH_ACRONYMS = {
    "API", "APIS", "URL", "URLS", "URI", "URIS", "HTTP", "HTTPS", "JSON",
    "YAML", "TOML", "XML", "HTML", "CSS", "SQL", "CSV", "TSV", "PDF",
    "MCP", "SDK", "SDKS", "CLI", "GUI", "IDE", "OS", "UI", "UX", "CPU",
    "GPU", "RAM", "ROM", "SSH", "FTP", "TCP", "UDP", "DNS", "IP", "ID",
    "IDS", "UUID", "GUID", "JWT", "OAUTH", "SAML", "CRUD", "REST",
    "GRPC", "RPC", "ABI", "AST", "JS", "TS", "PHP", "GO", "RB", "PY",
    "MD", "DOC", "DOCX", "PPT", "PPTX", "XLSX", "AI", "ML", "LLM",
    "LLMS", "NLP", "GPT", "ANSI", "UTF", "ASCII", "BOM", "EOL", "EOF",
    "FAQ", "FAQS", "TOC", "PR", "PRS", "MR", "MRS", "CI", "CD",
    "AWS", "GCP", "AZURE", "S3", "EC2", "TODO", "TODOS", "TBD",
    "FIXME", "XXX", "WIP", "NA", "OK", "OKAY", "IO", "FS", "DB", "DBS",
    "SSL", "TLS", "PNG", "JPG", "JPEG", "GIF", "SVG", "WEBP", "GIT",
    "VS", "OOP", "MIT", "BSD", "GPL", "LGPL", "APL", "POSIX", "BASH",
    "ZSH", "SH", "REPL", "JSON5", "JSONL", "NDJSON", "ROUGE", "BLEU",
    "F1", "NPM", "PIP", "GEM", "CARGO", "USA", "UK", "EU", "GMT", "UTC",
    "PST", "EST", "QA", "QC", "USD", "EUR", "GBP", "PASS", "FAIL",
    "PARTIAL", "GET", "POST", "PUT", "PATCH", "DELETE", "HEAD",
    "OPTIONS", "TRUE", "FALSE", "NULL", "NONE", "UNDEFINED", "NEW", "OLD",
}

CATEGORY_PREFIXES = [
    ("agent-prompt", "Agent prompt"),
    ("system-prompt", "System prompt"),
    ("system-reminder", "System reminder"),
    ("tool-description", "Tool description"),
    ("tool-parameter", "Tool parameter"),
    ("skill", "Skill"),
    ("data", "Data / template"),
]

# STANCE — five classes. The single mixed-polarity `evaluative` class has been
# split into `positive_evaluative` and `negative_evaluative`; positive/negative
# affect were previously folded together, masking sentiment polarity.
STANCE_MARKERS = {
    "directive": [
        "must", "should", "do not", "don't", "never", "always",
        "you must", "you must not", "you should", "you should not",
        "you need to", "you have to",
        "ensure", "make sure", "make sure to", "be sure to", "remember to",
        "it is mandatory",
    ],
    "expository": [
        "is", "are", "means", "refers to", "consists of", "is defined as",
        "represents", "is a", "are a", "this is", "these are",
        "in other words", "for example", "for instance", "specifically",
        "i.e.", "e.g.",
    ],
    "positive_evaluative": [
        "good", "better", "best", "preferred", "ideal", "optimal",
        "appropriate", "useful", "important", "critical", "essential",
        "valuable", "beneficial", "right", "correct",
        "excellent", "great", "helpful", "effective", "key",
        "superior", "recommended", "preferable", "suitable",
        "valid", "safe",
    ],
    "negative_evaluative": [
        "bad", "worse", "worst", "inappropriate", "useless", "trivial",
        "harmful", "problematic", "incorrect", "wrong",
        "broken", "suboptimal", "dangerous", "risky", "deprecated",
        "obsolete", "invalid", "unsuitable", "faulty", "flawed", "unsafe",
    ],
    "dialogic": [
        "you might", "you may", "you could", "consider", "perhaps",
        "however", "on the other hand", "alternatively", "in contrast",
        "but", "yet", "although", "while", "whereas",
        "do you", "what if", "suppose", "let's", "we can",
        "in our view", "we believe", "we think", "in my opinion",
    ],
}

REGISTER_MARKERS = {
    "frozen": [
        "hereby", "aforementioned", "wherein", "hereinafter",
        "notwithstanding", "shall not", "thereof", "thereto",
        "in accordance with", "pursuant to", "the party",
    ],
    "formal": [
        "moreover", "furthermore", "consequently", "therefore",
        "nevertheless", "accordingly", "thus", "hence",
        "it is necessary", "it should be noted", "as such",
        "regarding", "concerning", "with respect to", "with regard to",
        "in conclusion", "in summary",
    ],
    "consultative": [
        "please", "you can", "you may", "we recommend", "suggest",
        "consider", "if you", "when you", "in this case",
        "for example", "you might want to", "we suggest",
        "best practice", "tip:", "note:",
    ],
    "casual": [
        "let's", "here's", "you're", "we're", "i'm", "don't", "can't",
        "won't", "isn't", "aren't", "doesn't", "didn't",
        "yeah", "ok", "okay", "by the way", "anyway", "kind of",
        "sort of", "stuff", "thing",
    ],
}

# SENTENCE-LEVEL PRAGMATIC REGISTER — 4 lexicon classes. `imperative` and
# `directive` are NOT defined here; they reuse `classify_sent_mood` (cell 10)
# and `M_STANCE["directive"]` respectively. Near-zero pcts for `collaborative`
# and `appreciative` are an expected and informative finding for the Claude
# Code system-prompt corpus — absence is the signal.
SENTENCE_REGISTER_MARKERS = {
    "collaborative": [
        # Interpersonal first-person plural / shared-agency. Mostly absent
        # in this corpus by design — kept to surface that absence.
        "let's", "let us", "we should", "we can", "we will", "we'll",
        "we recommend", "we want", "we'd like", "together",
        "co-author", "pair-program", "you and i", "we agree", "our team",
    ],
    "permissive": [
        # Soft-directive / permission-granting / reader-agency. Attested.
        "please", "you can", "you may", "feel free to",
        "if you'd like", "if you prefer", "try to",
        "if possible", "optionally", "if needed",
    ],
    "appreciative": [
        # Gratitude / positive feedback. Mostly absent in this corpus by
        # design — kept to surface that absence.
        "thank you", "thanks", "appreciate", "appreciated",
        "great job", "well done", "excellent", "nice work",
        "awesome", "kudos", "bravo", "pleased", "good work",
        "fantastic", "terrific", "glad", "grateful", "much appreciated",
    ],
    "configuring": [
        # Config / parameter speech. Strongly attested in tool descriptions.
        "set to", "configure", "configured", "enable", "disable",
        "define", "defined", "specify", "specified",
        "default", "default to", "defaults to",
        "must be", "should be", "is required",
        "respond in", "respond with", "accepts", "expects",
        "output should",
    ],
}

# --- Modality detection: spaCy-only constants ----------------------------

# Bare-modal lemma → class. Fixed mapping (no head-verb disambiguation);
# ambiguous forms get reclassified by multi-word patterns in modality_for_doc.
SIMPLE_MODAL_LOOKUP = {
    "must":  "deontic",  "shall":  "deontic",  "should": "deontic", "ought": "deontic",
    "may":   "epistemic", "might":  "epistemic",
    "can":   "dynamic",   "could":  "dynamic",  "will":   "dynamic", "would":  "dynamic",
}

# Lemma sets driving multi-word modal detection in modality_for_doc.
EPISTEMIC_ADVERBS         = {"likely", "probably", "possibly", "perhaps",
                              "presumably", "apparently", "potentially"}
DEONTIC_LIGHT_VERBS       = {"have", "has", "had", "need", "needs", "needed",
                              "required", "ought"}
EPISTEMIC_LIGHT_VERBS     = {"seem", "seems", "appear", "appears"}
EPISTEMIC_AUX_AFTER_MODAL = {"be", "have"}   # MODAL + be/have → epistemic

print("VOCAB keys:        ", list(VOCAB.keys()))
print("STANCE_MARKERS:    ", list(STANCE_MARKERS.keys()))
print("REGISTER_MARKERS:  ", list(REGISTER_MARKERS.keys()))
print("SENTENCE_REGISTER: ", list(SENTENCE_REGISTER_MARKERS.keys()))
print(f"IMPERATIVE_MARKERS:    {len(IMPERATIVE_MARKERS)} entries")
print(f"CAPS_IMPERATIVE_TOKENS:{len(CAPS_IMPERATIVE_TOKENS)} entries")
print(f"JUSTIFICATION_PATTERNS:{len(JUSTIFICATION_PATTERNS)} regex patterns")
print(f"TECH_ACRONYMS:         {len(TECH_ACRONYMS)} entries")
print(f"SIMPLE_MODAL_LOOKUP:   {len(SIMPLE_MODAL_LOOKUP)} bare modals "
      f"({sorted(set(SIMPLE_MODAL_LOOKUP.values()))})")
print(f"EPISTEMIC_ADVERBS:     {sorted(EPISTEMIC_ADVERBS)}")
print(f"DEONTIC_LIGHT_VERBS:   {sorted(DEONTIC_LIGHT_VERBS)}")
print(f"EPISTEMIC_LIGHT_VERBS: {sorted(EPISTEMIC_LIGHT_VERBS)}")


VOCAB keys:         ['hard_prohibitions', 'hard_prescriptions', 'soft_prescriptions', 'politeness_direct', 'politeness_softening', 'warmth_encouragement', 'hedging', 'structural_markers', 'profanity', 'pronouns_2p', 'pronouns_1p']
STANCE_MARKERS:     ['directive', 'expository', 'positive_evaluative', 'negative_evaluative', 'dialogic']
REGISTER_MARKERS:   ['frozen', 'formal', 'consultative', 'casual']
SENTENCE_REGISTER:  ['collaborative', 'permissive', 'appreciative', 'configuring']
IMPERATIVE_MARKERS:    30 entries
CAPS_IMPERATIVE_TOKENS:23 entries
JUSTIFICATION_PATTERNS:32 regex patterns
TECH_ACRONYMS:         161 entries
SIMPLE_MODAL_LOOKUP:   10 bare modals (['deontic', 'dynamic', 'epistemic'])
EPISTEMIC_ADVERBS:     ['apparently', 'likely', 'perhaps', 'possibly', 'potentially', 'presumably', 'probably']
DEONTIC_LIGHT_VERBS:   ['had', 'has', 'have', 'need', 'needed', 'needs', 'ought', 'required']
EPISTEMIC_LIGHT_VERBS: ['appear', 'appears', 'seem', 'seems']


In [5]:
"""Load every .md file: parse HTML-comment frontmatter, strip code, categorize.

Frontmatter format (HTML-comment-wrapped YAML, all 287 files in this corpus):

    <!--
    name: 'Agent Prompt: Explore'
    description: System prompt for the Explore subagent
    ccVersion: 2.1.118
    variables:
      - GLOB_TOOL_NAME
    agentMetadata:
      agentType: 'Explore'
      tools:
        - *                  # this kind of unresolved YAML alias breaks safe_load
    -->

We extract ``name`` / ``description`` / ``ccVersion`` (top level) and
``agentMetadata.agentType`` (nested). If full YAML parsing fails on edge cases like
unresolved aliases (``*`` without anchor) or template variables, we fall back to
a line-by-line extraction of the known scalar top-level keys.
"""

import yaml as _yaml

HTML_FM_RE     = re.compile(r"^\s*<!--\s*\n(.*?)\n\s*-->\s*\n?", re.DOTALL)
CODE_FENCE_RE  = re.compile(r"```.*?```", re.DOTALL)
INLINE_CODE_RE = re.compile(r"`[^`]*`")
IMAGE_LINE_RE  = re.compile(r"^\s*!\[[^\]]*\]\([^)]*\)\s*$", re.MULTILINE)
LINK_ONLY_RE   = re.compile(r"^\s*\[[^\]]+\]\([^)]+\)\s*$", re.MULTILINE)

# Lenient line-by-line fallback for top-level scalar keys
SIMPLE_KV_RE = re.compile(r"^([A-Za-z_][\w.]*)\s*:\s*(.*?)\s*$")
NESTED_AGENTTYPE_RE = re.compile(
    r"^\s*agentType\s*:\s*['\"]?([A-Za-z][\w-]*)['\"]?\s*$", re.MULTILINE)

def _strip_yaml_quotes(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and ((s[0] == s[-1] == "'") or (s[0] == s[-1] == '"')):
        return s[1:-1]
    return s

def fallback_parse(yaml_text: str) -> dict:
    """Extract known top-level scalar keys without full YAML parse."""
    out = {}
    for raw_line in yaml_text.splitlines():
        # Only consider unindented top-level keys
        if raw_line[:1] in (" ", "\t"):
            continue
        m = SIMPLE_KV_RE.match(raw_line)
        if not m:
            continue
        key, val = m.group(1), _strip_yaml_quotes(m.group(2))
        if key in {"name", "description", "ccVersion"}:
            out[key] = val
    nested = NESTED_AGENTTYPE_RE.search(yaml_text)
    if nested:
        out["agentMetadata"] = {"agentType": nested.group(1)}
    return out

def parse_html_frontmatter(raw: str):
    """Extract HTML-comment-wrapped YAML frontmatter from start of doc.
    Returns (metadata_dict, body_without_frontmatter, used_fallback_bool)."""
    m = HTML_FM_RE.match(raw)
    if not m:
        return {}, raw, False
    yaml_text = m.group(1)
    used_fallback = False
    try:
        meta = _yaml.safe_load(yaml_text) or {}
        if not isinstance(meta, dict):
            meta = {}
    except _yaml.YAMLError:
        meta = {}
    if not meta:
        meta = fallback_parse(yaml_text)
        used_fallback = True
    return meta, raw[m.end():], used_fallback

def categorize(filename: str) -> str:
    name = filename.lower()
    for prefix, label in CATEGORY_PREFIXES:
        if name.startswith(prefix):
            return label
    return "Other"

def clean_markdown(raw: str) -> str:
    text = CODE_FENCE_RE.sub(" ", raw)
    text = INLINE_CODE_RE.sub(" ", text)
    text = IMAGE_LINE_RE.sub("", text)
    text = LINK_ONLY_RE.sub("", text)
    return text

records = []
fm_count = 0
fallback_count = 0
for path in tqdm(md_files, desc="loading"):
    raw = path.read_text(encoding="utf-8")
    meta, body, used_fallback = parse_html_frontmatter(raw)
    if meta:
        fm_count += 1
    if used_fallback:
        fallback_count += 1
    raw_text = clean_markdown(body)
    agent_meta = meta.get("agentMetadata") if isinstance(meta.get("agentMetadata"), dict) else {}
    records.append({
        "path":        path.name,
        "category":    categorize(path.name),
        "name":        meta.get("name", "") or "",
        "description": meta.get("description", "") or "",
        "ccVersion":   meta.get("ccVersion", "") or "",
        "agentType":   (agent_meta.get("agentType", "") if agent_meta else ""),
        "raw_text":    raw_text,
    })

df = pd.DataFrame(records)
df["clean_text"] = df["raw_text"].str.lower()
print(f"loaded {len(df)} prompts ({fm_count} with frontmatter, {fallback_count} via line-by-line fallback)")
print()
print("category distribution:")
print(df["category"].value_counts().to_string())
print()
print(f"unique ccVersions: {df['ccVersion'].nunique()}")
top_ver = df['ccVersion'].value_counts().head()
print(f"top 5 ccVersions: {top_ver.to_dict()}")
print(f"files with non-empty agentType: {(df['agentType'] != '').sum()}")
print()
print("--- sample frontmatter (first 5 files) ---")
print(df[["path", "name", "ccVersion", "agentType"]].head().to_string(index=False))


loading:   0%|          | 0/287 [00:00<?, ?it/s]

loaded 287 prompts (287 with frontmatter, 30 via line-by-line fallback)

category distribution:
category
Tool description    78
System prompt       63
System reminder     40
Data / template     38
Agent prompt        37
Skill               30
Tool parameter       1

unique ccVersions: 57
top 5 ccVersions: {'2.1.53': 47, '2.1.18': 24, '2.1.119': 20, '2.1.118': 20, '2.1.105': 13}
files with non-empty agentType: 6

--- sample frontmatter (first 5 files) ---
                                             path                                            name ccVersion agentType
         agent-prompt-agent-creation-architect.md          Agent Prompt: Agent creation architect    2.0.77          
          agent-prompt-auto-mode-rule-reviewer.md           Agent Prompt: Auto mode rule reviewer    2.1.81          
agent-prompt-background-agent-state-classifier.md Agent Prompt: Background agent state classifier   2.1.119          
  agent-prompt-bash-command-description-writer.md   Agent Prompt: Bas

In [6]:
"""Run the spaCy pipeline once. Cache Doc objects aligned with df.index."""
docs = list(tqdm(NLP.pipe(df["raw_text"].tolist(), batch_size=20),
                  total=len(df), desc="spaCy parse"))

df["n_tokens"] = [sum(1 for t in d if not t.is_space) for d in docs]
df["n_sents"]  = [sum(1 for _ in d.sents) for d in docs]

print(f"total tokens: {df['n_tokens'].sum():,}")
print(f"total sentences: {df['n_sents'].sum():,}")
print(f"mean tokens/file: {df['n_tokens'].mean():.0f}")
df[["path", "category", "n_tokens", "n_sents"]].head()


spaCy parse:   0%|          | 0/287 [00:00<?, ?it/s]

total tokens: 129,311
total sentences: 5,694
mean tokens/file: 451


,path,category,n_tokens,n_sents
0,agent-prompt-agent-creation-architect.md,Agent prompt,936,22
1,agent-prompt-auto-mode-rule-reviewer.md,Agent prompt,242,13
2,agent-prompt-background-agent-state-classifier.md,Agent prompt,719,27
3,agent-prompt-bash-command-description-writer.md,Agent prompt,180,5
4,agent-prompt-bash-command-prefix-detection.md,Agent prompt,630,22


## Analysis helpers

Build a `PhraseMatcher` per lexicon class and define small helpers for per-doc counting and per-1000-token normalization.

In [7]:
"""Lexicon → matcher helpers.

Two parallel unit normalisations are tracked for every density metric:

- ``pct(count, n_tokens)`` — count as a percentage of the file's word tokens
  (0-100). Answers: "what fraction of the prose is this feature?"

- ``per_sent(count, n_sents)`` — count divided by sentence count
  (raw rate, can be > 1). Answers: "how often per sentence does this feature
  appear?"

A high ``_pct`` with a low ``_per_sent`` indicates the feature is concentrated in
short matches (e.g. one-word emphatic markers); a high ``_per_sent`` with a low
``_pct`` indicates long sentences that each contain a few matches.
"""

from spacy.matcher import DependencyMatcher  # parse-tree cues for sentence_register

def build_phrase_matchers(nlp, lexicon: Dict[str, List[str]],
                          attr: str = "LOWER") -> Dict[str, PhraseMatcher]:
    matchers: Dict[str, PhraseMatcher] = {}
    for key, phrases in lexicon.items():
        m = PhraseMatcher(nlp.vocab, attr=attr)
        m.add(key, [nlp.make_doc(p) for p in phrases])
        matchers[key] = m
    return matchers

def count_matcher(doc, matcher: PhraseMatcher) -> int:
    return len(matcher(doc))

def pct(count: int, n_tokens: int) -> float:
    """Match count as a percentage of the file's word tokens (0-100)."""
    return round(100.0 * count / n_tokens, 4) if n_tokens else 0.0

def per_sent(count: int, n_sents: int) -> float:
    """Match count divided by sentence count (avg matches per sentence)."""
    return round(count / n_sents, 4) if n_sents else 0.0

# Build phrase matchers for every lexicon class. Modality is detected via
# spaCy parse patterns (see modality_for_doc), so it has no PhraseMatcher.
M_VOCAB         = build_phrase_matchers(NLP, VOCAB)
M_STANCE        = build_phrase_matchers(NLP, STANCE_MARKERS)
M_REGISTER      = build_phrase_matchers(NLP, REGISTER_MARKERS)
M_IMPERATIVE    = build_phrase_matchers(NLP, {"imperative_markers": IMPERATIVE_MARKERS})
M_SENT_REGISTER = build_phrase_matchers(NLP, SENTENCE_REGISTER_MARKERS)

# Caps imperative: keep ORIGINAL casing → match against orth on raw text Docs.
M_CAPS_IMP = PhraseMatcher(NLP.vocab, attr="ORTH")
M_CAPS_IMP.add("CAPS_IMP", [NLP.make_doc(p) for p in CAPS_IMPERATIVE_TOKENS])

# DependencyMatcher patterns for sentence_register parse-tree cues.
# Each pattern fires once per syntactic match; the analyzer (cell 16) buckets
# matches by sentence to set per-sentence flags.
DEP_MATCHER = DependencyMatcher(NLP.vocab)

# Collaborative cue: 1p-plural pronoun nsubj of a clause whose root has a
# modal/aux child. Catches "we should ship X", "we will run Y", etc.
DEP_MATCHER.add("COLLAB_WE_MODAL", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "modal",
     "RIGHT_ATTRS": {"DEP": "aux", "TAG": "MD"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "subj",
     "RIGHT_ATTRS": {"DEP": {"IN": ["nsubj", "nsubjpass"]},
                     "LEMMA": {"IN": ["we"]}}},
]])

# Permissive cue: "consider X-ing" — `consider` as head with a VBG xcomp child.
DEP_MATCHER.add("PERMISSIVE_CONSIDER_GERUND", [[
    {"RIGHT_ID": "head", "RIGHT_ATTRS": {"LEMMA": "consider"}},
    {"LEFT_ID": "head", "REL_OP": ">", "RIGHT_ID": "ger",
     "RIGHT_ATTRS": {"TAG": "VBG"}},
]])

# Configuring cue: parameter-noun nsubj + setter-verb root.
# Catches "the format must be JSON", "the parameter is set to ...", etc.
DEP_MATCHER.add("CONFIG_PARAM_SETTER", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {
        "POS": "VERB",
        "LEMMA": {"IN": ["be", "set", "equal", "default", "configure",
                          "specify", "accept", "return"]}}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "subj",
     "RIGHT_ATTRS": {
        "DEP": {"IN": ["nsubj", "nsubjpass"]},
        "LEMMA": {"IN": ["parameter", "setting", "response", "output", "format",
                          "value", "field", "key", "option", "argument"]}}},
]])

# Map DependencyMatcher rule IDs (string hashes) → sentence_register class.
DEP_RULE_TO_CLASS = {
    NLP.vocab.strings["COLLAB_WE_MODAL"]:           "collaborative",
    NLP.vocab.strings["PERMISSIVE_CONSIDER_GERUND"]: "permissive",
    NLP.vocab.strings["CONFIG_PARAM_SETTER"]:        "configuring",
}

print("matchers ready:",
      f"VOCAB={len(M_VOCAB)}, STANCE={len(M_STANCE)}, "
      f"REGISTER={len(M_REGISTER)}, IMPERATIVE={len(M_IMPERATIVE)}, "
      f"SENT_REGISTER={len(M_SENT_REGISTER)}, "
      f"DEP_MATCHER={len(DEP_RULE_TO_CLASS)} patterns")


matchers ready: VOCAB=11, STANCE=5, REGISTER=4, IMPERATIVE=1, SENT_REGISTER=4, DEP_MATCHER=3 patterns


## 1. Mood

Two signals reported side-by-side:

- **Grammatical** — per sentence, classify via the spaCy parse:
  - *Imperative*: root is a `VERB` whose dependents do not include `nsubj`/`nsubjpass`, AND the root is not preceded by an inverted auxiliary. Common imperative cue: the root verb is the first content token of the sentence.
  - *Interrogative*: sentence ends with `?` or shows aux-inversion (`AUX` precedes `nsubj`).
  - *Declarative*: everything else.
- **Lexical** — phrase-match `IMPERATIVE_MARKERS` to capture explicit cues like "you must", "no exceptions", "is required" that the parse-only heuristic misses.

In [8]:
"""Per-sentence mood classifier + per-file imperative-marker density.

This block now produces only the lexical-density signal (`marker_*`).
Per-sentence imperative classification used to live here; it has moved into
`metrics.sentence_register` (cell after this section). `classify_sent_mood`
remains as a public helper because the sentence_register analyzer reuses it
to drive the `imperative` per-sentence flag.
"""

def classify_sent_mood(sent) -> str:
    text = sent.text.strip()
    if not text:
        return "declarative"
    toks = [t for t in sent if not t.is_space]
    if not toks:
        return "declarative"
    if text.endswith("?"):
        return "interrogative"
    root = sent.root
    subj = next((c for c in root.children if c.dep_ in {"nsubj", "nsubjpass"}), None)
    aux  = next((c for c in root.children if c.dep_ in {"aux", "auxpass"}), None)
    if subj and aux and aux.i < subj.i and root.i > aux.i and text.endswith("."):
        pass
    elif aux and not subj and text.endswith("?"):
        return "interrogative"
    if root.pos_ == "VERB" and subj is None:
        first_content = next((t for t in toks if not t.is_punct), None)
        if first_content is not None and first_content.i <= root.i:
            return "imperative"
    return "declarative"

def mood_for_doc(doc, n_tokens: int, n_sents: int) -> dict:
    marker_count = count_matcher(doc, M_IMPERATIVE["imperative_markers"])
    return {
        "marker_count":    marker_count,
        "marker_pct":      pct(marker_count, n_tokens),
        "marker_per_sent": per_sent(marker_count, n_sents),
    }

mood_per_file = [mood_for_doc(d, n, s) for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_mood = pd.DataFrame(mood_per_file)
print("per-file mood (head):")
print(df_mood.head().to_string())
print()
print("category mean (mood — imperative-marker density):")
print(pd.concat([df[["category"]], df_mood], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())


per-file mood (head):
   marker_count  marker_pct  marker_per_sent
0             6      0.6410           0.2727
1             0      0.0000           0.0000
2             5      0.6954           0.1852
3             1      0.5556           0.2000
4             4      0.6349           0.1818

category mean (mood — imperative-marker density):
                  marker_count  marker_pct  marker_per_sent
category                                                   
Agent prompt             6.189       0.846            0.250
Data / template          4.368       0.584            0.126
Skill                    8.400       0.603            0.162
System prompt            2.222       1.093            0.260
System reminder          1.675       1.023            0.230
Tool description         2.205       2.136            0.411
Tool parameter           0.000       0.000            0.000


## 2. Register

Quantitative + lexical-class register profile.

- **Type-token ratio** (TTR) on lemmatized content tokens (no punct/space/stop).
- **Mean sentence length** in tokens.
- **Mean dependency depth** = average distance from each token to its sentence root.
- **F-score** (Heylighen & Dewaele): `F = 50 + 50 * (formal_pos − informal_pos) / total`
  with `formal_pos = NOUN+ADJ+ADP+DET` and `informal_pos = PRON+VERB+ADV+INTJ`.
- **Lexical register class** — phrase-match the four `REGISTER_MARKERS` lists, normalise as
  **% of the file's tokens**, and assign a `dominant` register (argmax across the four).

In [9]:
"""Register metrics: TTR, sentence length, dep-depth, F-score, four register classes
(both per-word and per-sentence)."""

from collections import deque

FORMAL_POS   = {"NOUN", "ADJ", "ADP", "DET"}
INFORMAL_POS = {"PRON", "VERB", "ADV", "INTJ"}

def doc_dep_depths(doc) -> List[int]:
    depths = [0] * len(doc)
    for sent in doc.sents:
        q = deque([(sent.root, 0)])
        while q:
            tok, d = q.popleft()
            depths[tok.i - doc[0].i] = d
            for child in tok.children:
                q.append((child, d + 1))
    return depths

def register_for_doc(doc, n_tokens: int, n_sents: int) -> dict:
    content_lemmas = []
    formal = informal = 0
    for t in doc:
        if t.is_space:
            continue
        if not t.is_punct and not t.is_stop:
            content_lemmas.append(t.lemma_.lower())
        if t.pos_ in FORMAL_POS:
            formal += 1
        elif t.pos_ in INFORMAL_POS:
            informal += 1

    ttr = round(len(set(content_lemmas)) / len(content_lemmas), 4) if content_lemmas else 0.0

    sents = [s for s in doc.sents if s.text.strip()]
    if sents:
        sent_lens = [sum(1 for t in s if not t.is_space) for s in sents]
        mean_sent_len = round(sum(sent_lens) / len(sents), 2)
    else:
        mean_sent_len = 0.0

    depths = doc_dep_depths(doc)
    mean_dep_depth = round(sum(depths) / max(len(depths), 1), 3) if depths else 0.0

    total_pos = formal + informal
    f_score = round(50 + 50 * (formal - informal) / total_pos, 2) if total_pos else 50.0

    reg_counts = {key: count_matcher(doc, m) for key, m in M_REGISTER.items()}
    out = {
        "ttr": ttr,
        "mean_sent_len": mean_sent_len,
        "dep_depth": mean_dep_depth,
        "f_score": f_score,
        **{f"{k}_count":    v                      for k, v in reg_counts.items()},
        **{f"{k}_pct":      pct(v, n_tokens)       for k, v in reg_counts.items()},
        **{f"{k}_per_sent": per_sent(v, n_sents)   for k, v in reg_counts.items()},
        "dominant_register": (max(reg_counts, key=reg_counts.get)
                              if reg_counts and max(reg_counts.values()) > 0 else "none"),
    }
    return out

register_per_file = [register_for_doc(d, n, s)
                      for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_register = pd.DataFrame(register_per_file)
print("per-file register (head):")
print(df_register.head().to_string())
print()
print("category mean (numeric register cols):")
num_cols = [c for c in df_register.columns
            if c != "dominant_register" and not c.endswith("_count")]
print(pd.concat([df[["category"]], df_register[num_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())


per-file register (head):
      ttr  mean_sent_len  dep_depth  f_score  frozen_count  formal_count  consultative_count  casual_count  frozen_pct  formal_pct  consultative_pct  casual_pct  frozen_per_sent  formal_per_sent  consultative_per_sent  casual_per_sent dominant_register
0  0.5407          42.55      3.910    71.98             0             0                   5             1         0.0         0.0            0.5342      0.1068              0.0              0.0                 0.2273           0.0455      consultative
1  0.6421          18.62      2.661    75.89             0             0                   0             1         0.0         0.0            0.0000      0.4132              0.0              0.0                 0.0000           0.0769            casual
2  0.6502          26.63      2.894    66.22             0             0                   3             7         0.0         0.0            0.4172      0.9736              0.0              0.0                 0.11

## 3. Stance

Four-class lexical stance via `STANCE_MARKERS` (`directive` / `expository` / `evaluative` /
`dialogic`), plus first- and second-person engagement densities from `VOCAB.pronouns_1p` /
`pronouns_2p`. All values are **% of the file's own tokens**.

In [10]:
"""Stance: five-class densities + 1p/2p engagement (per-word and per-sentence).

The legacy single `evaluative` class has been split into `positive_evaluative`
and `negative_evaluative` so sentiment polarity is a first-class signal rather
than a mixed aggregate.
"""

def stance_for_doc(doc, n_tokens: int, n_sents: int) -> dict:
    counts = {key: count_matcher(doc, m) for key, m in M_STANCE.items()}
    p1p = count_matcher(doc, M_VOCAB["pronouns_1p"])
    p2p = count_matcher(doc, M_VOCAB["pronouns_2p"])
    out: dict = {}
    for k, v in counts.items():
        out[f"{k}_count"]    = v
        out[f"{k}_pct"]      = pct(v, n_tokens)
        out[f"{k}_per_sent"] = per_sent(v, n_sents)
    out["pronouns_1p_count"]    = p1p
    out["pronouns_1p_pct"]      = pct(p1p, n_tokens)
    out["pronouns_1p_per_sent"] = per_sent(p1p, n_sents)
    out["pronouns_2p_count"]    = p2p
    out["pronouns_2p_pct"]      = pct(p2p, n_tokens)
    out["pronouns_2p_per_sent"] = per_sent(p2p, n_sents)
    out["dominant_stance"] = (max(counts, key=counts.get)
                              if counts and max(counts.values()) > 0 else "none")
    return out

stance_per_file = [stance_for_doc(d, n, s)
                    for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_stance = pd.DataFrame(stance_per_file)
print("per-file stance (head):")
print(df_stance.head().to_string())
print()
print("dominant stance by category:")
print(pd.crosstab(df["category"], df_stance["dominant_stance"]))
print()
print("category mean stance (% of words and per-sentence rate):")
num_cols = [c for c in df_stance.columns
            if c != "dominant_stance" and not c.endswith("_count")]
print(pd.concat([df[["category"]], df_stance[num_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())


per-file stance (head):
   directive_count  directive_pct  directive_per_sent  expository_count  expository_pct  expository_per_sent  positive_evaluative_count  positive_evaluative_pct  positive_evaluative_per_sent  negative_evaluative_count  negative_evaluative_pct  negative_evaluative_per_sent  dialogic_count  dialogic_pct  dialogic_per_sent  pronouns_1p_count  pronouns_1p_pct  pronouns_1p_per_sent  pronouns_2p_count  pronouns_2p_pct  pronouns_2p_per_sent dominant_stance
0               18         1.9231              0.8182                15          1.6026               0.6818                          6                   0.6410                        0.2727                          0                   0.0000                        0.0000               3        0.3205             0.1364                  2           0.2137                0.0909                 16           1.7094                0.7273       directive
1                3         1.2397              0.2308               

## 3b. Sentence-level pragmatic register

A per-sentence multi-label classifier answering: **what speech act is each sentence performing?** This is a pragmatic dimension distinct from formality (`register`) and commitment (`stance`).

Six classes are tracked, with multi-label assignment (a single sentence may carry several labels):

| Class | Detector | Expected in this corpus |
|---|---|---|
| **`imperative`** | `classify_sent_mood(sent) == "imperative"` (cell 10 reused) | High — directive prompts |
| **`directive`** | overlaps any `STANCE_MARKERS["directive"]` span | High |
| **`collaborative`** | `M_SENT_REGISTER["collaborative"]` OR `we + modal + V` DependencyMatcher | **Near-zero — that's a finding.** Claude Code system prompts are functional/directive, not interpersonal. |
| **`permissive`** | `M_SENT_REGISTER["permissive"]` OR `consider X-ing` DependencyMatcher | Modest (`please`, `you can`, `you may`) |
| **`appreciative`** | `M_SENT_REGISTER["appreciative"]` | **Near-zero — that's a finding.** Same reason. |
| **`configuring`** | `M_SENT_REGISTER["configuring"]` OR `param-noun + setter-verb` DependencyMatcher | High in tool descriptions |

A sentence with none of the six flags is `none`; `none_sent_pct` is mutually exclusive with the rest. The four content `*_sent_pct` values can sum to >100% (intentional — multi-label).

**Why we keep the near-zero classes.** Knowing what a corpus *isn't* doing is as important as knowing what it is. A near-zero `appreciative_sent_pct` across 287 prompts is a structural fact about how Claude Code authors write — not a measurement to be hidden by lexicon pruning.

**spaCy ecosystem leverage.** `spacy.matcher.DependencyMatcher` (cell 8) replaces hand-rolled token loops for the three parse-tree cues. `Token.morph` (`Mood=Imp`) was considered and rejected — unreliable in `en_core_web_sm`; `classify_sent_mood`'s parse-tree heuristic is better tuned. External sentiment lexicons (VADER, Hu-Liu, spacytextblob) were considered and rejected — hand-curated round-trippable lexicons preserve audit transparency.

**Cross-axis overlaps (intentional).**
- `please` lives in `SENTENCE_REGISTER_MARKERS["permissive"]` AND `REGISTER_MARKERS["consultative"]`. Both fire — different axes.
- `must`/`should` drive both `STANCE_MARKERS["directive"]` (commitment density) and the `directive` per-sentence flag (matcher reuse — guaranteed agreement).
- `excellent`/`important` drive `STANCE_MARKERS["positive_evaluative"]` (doc density) AND `SENTENCE_REGISTER_MARKERS["appreciative"]` (per-sentence flag). Different abstraction levels.

In [11]:
"""Sentence-level pragmatic register: 6-class multi-label per-sentence classifier.

Per Doc:
  1. Run all 4 PhraseMatchers from M_SENT_REGISTER + the directive sub-matcher of
     M_STANCE + DEP_MATCHER once each.
  2. Bucket every match's first token into its containing sentence (`token.sent`).
  3. Iterate sentences and emit the 6 flags + `none`.

Cross-check: sentence_register.imperative_sent_count must equal the number of
sentences for which classify_sent_mood returned "imperative". Asserted below.
"""

# Class order is the dominant tie-break order (collaborative > permissive >
# appreciative > imperative > directive > configuring).
SENT_REGISTER_CLASSES = ["collaborative", "permissive", "appreciative",
                          "imperative", "directive", "configuring"]


def _bucket_matches_by_sent(doc, span_iter):
    """Bucket each (start_token_idx, label) into the sentence it falls in.

    Returns: dict[sent_index_in_doc -> set[label]].
    """
    out: Dict[int, set] = defaultdict(set)
    sent_starts = [s.start for s in doc.sents]
    sent_index_for: Dict[int, int] = {}  # cached token-idx -> sent-idx
    for s_idx, sent in enumerate(doc.sents):
        for tok_i in range(sent.start, sent.end):
            sent_index_for[tok_i] = s_idx
    for start_tok, label in span_iter:
        s_idx = sent_index_for.get(start_tok)
        if s_idx is not None:
            out[s_idx].add(label)
    return out


def sentence_register_for_doc(doc, n_sents: int) -> dict:
    # 1. Collect all spans (PhraseMatcher hits) with their class label.
    spans = []  # (start_token_idx, label)
    for cls, matcher in M_SENT_REGISTER.items():
        for _, start, _end in matcher(doc):
            spans.append((start, cls))
    # Directive flag reuses the existing stance directive matcher.
    for _, start, _end in M_STANCE["directive"](doc):
        spans.append((start, "directive"))
    # DependencyMatcher hits — each match is (rule_id, [token_ids...]).
    for rule_id, token_ids in DEP_MATCHER(doc):
        cls = DEP_RULE_TO_CLASS.get(rule_id)
        if cls is not None and token_ids:
            spans.append((token_ids[0], cls))

    sent_labels = _bucket_matches_by_sent(doc, spans)

    # 2. Iterate sentences; add imperative flag from classify_sent_mood.
    counts = {cls: 0 for cls in SENT_REGISTER_CLASSES}
    none_count = 0
    seen_sents = 0
    imperative_via_mood = 0
    for s_idx, sent in enumerate(doc.sents):
        if not sent.text.strip():
            continue
        seen_sents += 1
        labels = set(sent_labels.get(s_idx, ()))
        if classify_sent_mood(sent) == "imperative":
            labels.add("imperative")
            imperative_via_mood += 1
        if labels:
            for cls in labels:
                if cls in counts:
                    counts[cls] += 1
        else:
            none_count += 1

    # 3. Build output dict.
    denom = seen_sents or n_sents or 1
    out: dict = {}
    for cls in SENT_REGISTER_CLASSES:
        out[f"{cls}_sent_count"] = counts[cls]
        out[f"{cls}_sent_pct"]   = round(100.0 * counts[cls] / denom, 4)
    out["none_sent_count"] = none_count
    out["none_sent_pct"]   = round(100.0 * none_count / denom, 4)
    # Dominant: highest count among the six classes; tie-break = class order.
    nonzero = [(cls, counts[cls]) for cls in SENT_REGISTER_CLASSES if counts[cls] > 0]
    out["dominant"] = max(nonzero, key=lambda kv: kv[1])[0] if nonzero else ""

    # Cross-check (development assertion; harmless to leave in production).
    assert out["imperative_sent_count"] == imperative_via_mood, (
        f"imperative_sent_count {out['imperative_sent_count']} != mood {imperative_via_mood}")
    return out


sentence_register_per_file = [
    sentence_register_for_doc(d, s)
    for d, s in zip(docs, df["n_sents"])
]
df_sent_register = pd.DataFrame(sentence_register_per_file)

print("per-file sentence_register (head):")
print(df_sent_register.head().to_string())
print()
print("dominant by category:")
print(pd.crosstab(df["category"], df_sent_register["dominant"]))
print()
print("category mean sentence_register (% of sentences):")
pct_cols = [c for c in df_sent_register.columns if c.endswith("_sent_pct")]
print(pd.concat([df[["category"]], df_sent_register[pct_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(2).to_string())


per-file sentence_register (head):
   collaborative_sent_count  collaborative_sent_pct  permissive_sent_count  permissive_sent_pct  appreciative_sent_count  appreciative_sent_pct  imperative_sent_count  imperative_sent_pct  directive_sent_count  directive_sent_pct  configuring_sent_count  configuring_sent_pct  none_sent_count  none_sent_pct    dominant
0                         0                  0.0000                      3              13.6364                        0                    0.0                      7              31.8182                     9             40.9091                       5               22.7273                7        31.8182   directive
1                         0                  0.0000                      0               0.0000                        0                    0.0                      1               7.6923                     2             15.3846                       1                7.6923               10        76.9231   directive
2    

## 4. Modality

Single spaCy-based detector. We classify every modal expression into **deontic** (obligation),
**epistemic** (inference / probability), or **dynamic** (ability / future) by walking the spaCy
parse and applying these patterns:

1. **Bare modal auxiliary** — single tokens with `tag_ == "MD"` mapped via `SIMPLE_MODAL_LOOKUP`
   (`should`/`must`/`shall`/`ought` → deontic, `may`/`might` → epistemic,
   `can`/`could`/`will`/`would` → dynamic).
2. **Modal + `be` / `have`** — `should be`, `must be`, `may have`, `could be`, `would be`,
   `could have`, `would have`, `may be` → reclassified as **epistemic** (these forms express
   inference, not obligation).
3. **Light deontic verbs** + `to` + VERB — `have to`, `need to`, `required to`, `ought to` →
   **deontic**.
4. **Light epistemic verbs** + `to` + VERB — `seems to`, `appears to` → **epistemic**.
5. **`able` + `to` + VERB** (any preceding `be`-form: `is able to`, `was able to`, …) → **dynamic**.
6. **Epistemic adverbs** — tokens with `pos_ == "ADV"` and lemma in `EPISTEMIC_ADVERBS`
   (`likely`, `probably`, `possibly`, `perhaps`, `presumably`, `apparently`, `potentially`) →
   **epistemic**.

Every match is counted once. The detector also tracks the surface form of every matched span and
returns the most frequent as `top_construction`.

In [12]:
"""Modality detector — pure spaCy parse-tree based, no PhraseMatcher.

For every Doc we walk tokens left-to-right, applying the patterns documented in the
markdown cell above. Each pattern:
- Counts the modal expression once into one of {deontic, epistemic, dynamic}.
- Records the surface form of the matched span so we can report the most frequent
  ``top_construction`` per file.
"""

def _has_to_verb_after(doc, i: int) -> bool:
    """Return True if doc[i+1] is `to` and doc[i+2] is a VERB / AUX."""
    return (
        i + 2 < len(doc)
        and doc[i + 1].lemma_.lower() == "to"
        and doc[i + 2].pos_ in {"VERB", "AUX"}
    )

def modality_for_doc(doc, n_tokens: int, n_sents: int) -> dict:
    counts = Counter()
    constructions = Counter()
    consumed: set[int] = set()

    for t in doc:
        if t.i in consumed or t.is_space:
            continue
        lemma = t.lemma_.lower()

        # 1. Bare modal auxiliary (MD tag)
        if t.tag_ == "MD":
            nxt = doc[t.i + 1] if t.i + 1 < len(doc) else None
            # 2. MODAL + be/have → epistemic
            if nxt is not None and nxt.lemma_.lower() in EPISTEMIC_AUX_AFTER_MODAL:
                counts["epistemic"] += 1
                constructions[f"{t.text.lower()} {nxt.text.lower()}"] += 1
                consumed.add(t.i)
                continue
            cls = SIMPLE_MODAL_LOOKUP.get(lemma, "dynamic")
            counts[cls] += 1
            constructions[t.text.lower()] += 1
            consumed.add(t.i)
            continue

        # 3. Light deontic verb + to + VERB
        if lemma in DEONTIC_LIGHT_VERBS and _has_to_verb_after(doc, t.i):
            counts["deontic"] += 1
            constructions[f"{t.text.lower()} to"] += 1
            consumed.update({t.i, t.i + 1})
            continue

        # 4. Light epistemic verb + to + VERB
        if lemma in EPISTEMIC_LIGHT_VERBS and _has_to_verb_after(doc, t.i):
            counts["epistemic"] += 1
            constructions[f"{t.text.lower()} to"] += 1
            consumed.update({t.i, t.i + 1})
            continue

        # 5. "able + to + VERB" → dynamic
        if lemma == "able" and _has_to_verb_after(doc, t.i):
            counts["dynamic"] += 1
            constructions["able to"] += 1
            consumed.add(t.i)
            continue

        # 6. Epistemic adverb
        if t.pos_ == "ADV" and lemma in EPISTEMIC_ADVERBS:
            counts["epistemic"] += 1
            constructions[lemma] += 1
            consumed.add(t.i)
            continue

    top = constructions.most_common(1)[0][0] if constructions else None

    out: dict = {}
    for cls in ("deontic", "epistemic", "dynamic"):
        c = counts.get(cls, 0)
        out[f"{cls}_count"]    = c
        out[f"{cls}_pct"]      = pct(c, n_tokens)
        out[f"{cls}_per_sent"] = per_sent(c, n_sents)
    out["top_construction"] = top
    return out

modality_per_file = [modality_for_doc(d, n, s)
                      for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_modality = pd.DataFrame(modality_per_file)

print("per-file modality (head):")
print(df_modality.head().to_string())
print()
print("category means (% of words and per-sentence rate):")
key_cols = ["deontic_pct", "deontic_per_sent",
            "epistemic_pct", "epistemic_per_sent",
            "dynamic_pct", "dynamic_per_sent"]
print(pd.concat([df[["category"]], df_modality[key_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())
print()
print("corpus-wide modality counts:")
totals = df_modality[["deontic_count", "epistemic_count", "dynamic_count"]].sum()
print(totals.to_string())


per-file modality (head):
   deontic_count  deontic_pct  deontic_per_sent  epistemic_count  epistemic_pct  epistemic_per_sent  dynamic_count  dynamic_pct  dynamic_per_sent top_construction
0              5       0.5342            0.2273                8         0.8547              0.3636              4       0.4274            0.1818           should
1              2       0.8264            0.1538                2         0.8264              0.1538              2       0.8264            0.1538           should
2              2       0.2782            0.0741                0         0.0000              0.0000              6       0.8345            0.2222               ca
3              0       0.0000            0.0000                0         0.0000              0.0000              0       0.0000            0.0000              NaN
4              1       0.1587            0.0455                5         0.7937              0.2273              4       0.6349            0.1818             w

## 5. Vocabulary profile

Phrase-match every key in `VOCAB` against each Doc and record raw counts plus per-1000-token rates.

In [13]:
"""VOCAB scan — counts, % of words, and per-sentence rates for every key."""

VOCAB_KEYS = list(VOCAB.keys())

def vocab_for_doc(doc, n_tokens: int, n_sents: int) -> dict:
    out: dict = {}
    for key, m in M_VOCAB.items():
        c = count_matcher(doc, m)
        out[f"{key}_count"]    = c
        out[f"{key}_pct"]      = pct(c, n_tokens)
        out[f"{key}_per_sent"] = per_sent(c, n_sents)
    return out

vocab_per_file = [vocab_for_doc(d, n, s)
                   for d, n, s in zip(docs, df["n_tokens"], df["n_sents"])]
df_vocab = pd.DataFrame(vocab_per_file)

pct_cols      = [f"{k}_pct" for k in VOCAB_KEYS]
per_sent_cols = [f"{k}_per_sent" for k in VOCAB_KEYS]

print("category mean (% of words):")
print(pd.concat([df[["category"]], df_vocab[pct_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())
print()
print("category mean (matches per sentence):")
print(pd.concat([df[["category"]], df_vocab[per_sent_cols]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())
print()
total = df_vocab[[f"{k}_count" for k in VOCAB_KEYS]].sum()
print("corpus-wide raw counts:")
for k in VOCAB_KEYS:
    print(f"  {k:24s} {int(total[f'{k}_count']):6d}")


category mean (% of words):
                  hard_prohibitions_pct  hard_prescriptions_pct  soft_prescriptions_pct  politeness_direct_pct  politeness_softening_pct  warmth_encouragement_pct  hedging_pct  structural_markers_pct  profanity_pct  pronouns_2p_pct  pronouns_1p_pct
category                                                                                                                                                                                                                                                
Agent prompt                      0.517                   0.265                   0.393                  0.043                     0.038                     0.007        0.171                   0.278            0.0            1.317            0.073
Data / template                   0.310                   0.269                   0.033                  0.008                     0.045                     0.000        0.093                   0.152            0.0           

## 6. ALL CAPS emphasis

Tokens of ≥2 letters that are **entirely** uppercase, with `TECH_ACRONYMS` excluded.
This isolates emphatic shouting (`IMPORTANT`, `NEVER`, `MUST`) from legitimate acronyms
(`API`, `JSON`, `URL`).

In [14]:
"""ALL CAPS detection on raw_text. Excludes TECH_ACRONYMS."""

ALLCAPS_RE = re.compile(r"\b[A-Z]{2,}(?:[-_'][A-Z]{2,})*\b")

def all_caps_for_text(text: str, n_tokens: int, n_sents: int) -> dict:
    matches = ALLCAPS_RE.findall(text)
    filtered = [m for m in matches if m not in TECH_ACRONYMS]
    counts = Counter(filtered)
    return {
        "count":    len(filtered),
        "distinct": len(counts),
        "pct":      pct(len(filtered), n_tokens),
        "per_sent": per_sent(len(filtered), n_sents),
        "top":      [{"token": t, "count": c} for t, c in counts.most_common(3)],
    }

all_caps_per_file = [all_caps_for_text(t, n, s)
                      for t, n, s in zip(df["raw_text"], df["n_tokens"], df["n_sents"])]
df_all_caps = pd.DataFrame(all_caps_per_file)
print("per-file ALL CAPS (head):")
print(df_all_caps.head().to_string())
print()
print("category mean ALL CAPS (% of words / per sentence):")
print(pd.concat([df[["category"]], df_all_caps[["count", "pct", "per_sent"]]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

corpus_all_caps = Counter()
for txt in df["raw_text"]:
    for tok in ALLCAPS_RE.findall(txt):
        if tok not in TECH_ACRONYMS:
            corpus_all_caps[tok] += 1
print()
print("corpus-wide top-25 ALL CAPS tokens:")
for tok, c in corpus_all_caps.most_common(25):
    print(f"  {tok:20s} {c}")


per-file ALL CAPS (head):
   count  distinct     pct  per_sent                                                                                                                                      top
0      6         3  0.6410    0.2727                                [{'token': 'CLAUDE', 'count': 3}, {'token': 'TASK_TOOL_NAME', 'count': 2}, {'token': 'NOTE', 'count': 1}]
1      0         0  0.0000    0.0000                                                                                                                                       []
2     15        15  2.0862    0.5556  [{'token': 'STATES', 'count': 1}, {'token': 'BACKGROUND_AGENT_STATE_DEFINITIONS', 'count': 1}, {'token': 'DISAMBIGUATION', 'count': 1}]
3      0         0  0.0000    0.0000                                                                                                                                       []
4     19         9  3.0159    0.8636                                      [{'token': 'GOEXPERIMENT', 'co

## 7. CAPS imperative tokens

Direct phrase match on `CAPS_IMPERATIVE_TOKENS` against `raw_text` (case-sensitive). This is
the "shouting register" — explicit cues like `IMPORTANT`, `NEVER`, `MUST NOT`, `CRITICAL`,
`WARNING`. A separate signal from generic ALL CAPS (which also catches proper nouns and template
variables).

In [15]:
"""CAPS imperative tokens — phrase-match against raw_text. Word-boundary safe via regex
to handle multi-word phrases like 'MUST NOT' / 'DO NOT' / 'VERY IMPORTANT'."""

sorted_caps = sorted(CAPS_IMPERATIVE_TOKENS, key=len, reverse=True)
CAPS_IMP_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(p) for p in sorted_caps) + r")\b"
)

def caps_imperative_for_text(text: str, n_tokens: int, n_sents: int) -> dict:
    hits = CAPS_IMP_RE.findall(text)
    counts = Counter(hits)
    return {
        "count":    len(hits),
        "pct":      pct(len(hits), n_tokens),
        "per_sent": per_sent(len(hits), n_sents),
        "hits":     dict(counts.most_common()),
    }

caps_imperative_per_file = [
    caps_imperative_for_text(t, n, s)
    for t, n, s in zip(df["raw_text"], df["n_tokens"], df["n_sents"])
]
df_caps_imperative = pd.DataFrame(caps_imperative_per_file)
print("per-file CAPS imperative (head):")
print(df_caps_imperative.head().to_string())
print()
print("category mean CAPS imperative (% of words / per sentence):")
print(pd.concat([df[["category"]], df_caps_imperative[["count", "pct", "per_sent"]]], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())

corpus_caps_imperative = Counter()
for txt in df["raw_text"]:
    corpus_caps_imperative.update(CAPS_IMP_RE.findall(txt))
print()
print("corpus-wide CAPS imperative frequency:")
for tok in CAPS_IMPERATIVE_TOKENS:
    if corpus_caps_imperative[tok]:
        print(f"  {tok:18s} {corpus_caps_imperative[tok]}")


per-file CAPS imperative (head):
   count     pct  per_sent              hits
0      1  0.1068    0.0455       {'NOTE': 1}
1      0  0.0000    0.0000                {}
2      0  0.0000    0.0000                {}
3      0  0.0000    0.0000                {}
4      1  0.1587    0.0455  {'IMPORTANT': 1}

category mean CAPS imperative (% of words / per sentence):
                  count    pct  per_sent
category                                
Agent prompt      1.324  0.203     0.073
Data / template   0.026  0.002     0.000
Skill             0.367  0.013     0.004
System prompt     0.381  0.163     0.048
System reminder   0.475  0.297     0.063
Tool description  0.615  0.353     0.080
Tool parameter    0.000  0.000     0.000

corpus-wide CAPS imperative frequency:
  IMPORTANT          37
  VERY IMPORTANT     1
  CRITICAL           17
  MANDATORY          3
  REQUIRED           4
  MUST               21
  MUST NOT           3
  NEVER              27
  ALWAYS             7
  DO NOT         

## 8. Justification patterns

For each regex in `JUSTIFICATION_PATTERNS` count matches in `clean_text` (case-insensitive).
Sum to a total justification count and compute:

**Justification ratio** = `justification_count / (imperative_marker_count + 1)`

A high ratio means rules are routinely paired with a stated reason (teaching tone). A low ratio
means rules without explanations (commanding tone).

In [16]:
"""Justification patterns + ratio (per-word and per-sentence)."""

JUST_RE = [re.compile(p, re.IGNORECASE) for p in JUSTIFICATION_PATTERNS]

def justification_for_text(text: str, n_tokens: int, n_sents: int,
                            marker_count: int) -> dict:
    total = sum(len(r.findall(text)) for r in JUST_RE)
    return {
        "count":    total,
        "pct":      pct(total, n_tokens),
        "per_sent": per_sent(total, n_sents),
        "ratio":    round(total / (marker_count + 1), 3),
    }

justification_per_file = [
    justification_for_text(t, n, s, m)
    for t, n, s, m in zip(df["clean_text"], df["n_tokens"], df["n_sents"],
                           df_mood["marker_count"])
]
df_justification = pd.DataFrame(justification_per_file)
print("per-file justification (head):")
print(df_justification.head().to_string())
print()
print("category mean (justification, % of words / per sentence):")
print(pd.concat([df[["category"]], df_justification], axis=1)
        .groupby("category").mean(numeric_only=True).round(3).to_string())


per-file justification (head):
   count     pct  per_sent  ratio
0      6  0.6410    0.2727  0.857
1      1  0.4132    0.0769  1.000
2      1  0.1391    0.0370  0.167
3      1  0.5556    0.2000  0.500
4      5  0.7937    0.2273  1.000

category mean (justification, % of words / per sentence):
                  count    pct  per_sent  ratio
category                                       
Agent prompt      1.703  0.288     0.076  0.295
Data / template   0.974  0.114     0.021  0.193
Skill             2.633  0.273     0.083  0.377
System prompt     0.794  0.299     0.073  0.244
System reminder   0.375  0.199     0.042  0.089
Tool description  0.731  0.343     0.077  0.146
Tool parameter    0.000  0.000     0.000  0.000


## 9. Assemble per-file metrics

Collect all per-file metrics into a list of nested dicts (one per `.md` file) ready for YAML
serialisation. Keys are stable: `mood`, `register`, `stance`, `modality`, `vocab`, `all_caps`,
`caps_imperative`, `justification`.

In [17]:
"""Build the nested per-file metric tree.

Field-naming contract (applied uniformly across every block):
- ``count`` — raw integer
- ``pct`` — % of word tokens (0-100)
- ``per_sent`` — average matches per sentence (rate)
- ``sent_pct`` — % of sentences (used only by sentence_register)
"""

def build_file_record(i: int) -> dict:
    row = df.iloc[i]
    return {
        "path":        row["path"],
        "category":    row["category"],
        "name":        row["name"] or None,
        "description": row["description"] or None,
        "ccVersion":   row["ccVersion"] or None,
        "agentType":   row["agentType"] or None,
        "n_tokens":    int(row["n_tokens"]),
        "n_sents":     int(row["n_sents"]),
        "metrics": {
            "mood":              mood_per_file[i],
            "register":          register_per_file[i],
            "stance":            stance_per_file[i],
            "sentence_register": sentence_register_per_file[i],
            "modality":          modality_per_file[i],
            "vocab": {
                k: {"count":    int(vocab_per_file[i][f"{k}_count"]),
                    "pct":      vocab_per_file[i][f"{k}_pct"],
                    "per_sent": vocab_per_file[i][f"{k}_per_sent"]}
                for k in VOCAB_KEYS
            },
            "all_caps":        all_caps_per_file[i],
            "caps_imperative": caps_imperative_per_file[i],
            "justification":   justification_per_file[i],
        },
    }

per_file_records = [build_file_record(i) for i in range(len(df))]
print(f"built {len(per_file_records)} per-file records")
print()
print("--- sample (file 0):")
import pprint
pprint.pp(per_file_records[0], depth=3, sort_dicts=False)


built 287 per-file records

--- sample (file 0):
{'path': 'agent-prompt-agent-creation-architect.md',
 'category': 'Agent prompt',
 'name': 'Agent Prompt: Agent creation architect',
 'description': 'System prompt for creating custom AI agents with detailed '
                'specifications',
 'ccVersion': '2.0.77',
 'agentType': None,
 'n_tokens': 936,
 'n_sents': 22,
 'metrics': {'mood': {'marker_count': 6,
                      'marker_pct': 0.641,
                      'marker_per_sent': 0.2727},
             'register': {'ttr': 0.5407,
                          'mean_sent_len': 42.55,
                          'dep_depth': 3.91,
                          'f_score': 71.98,
                          'frozen_count': 0,
                          'formal_count': 0,
                          'consultative_count': 5,
                          'casual_count': 1,
                          'frozen_pct': 0.0,
                          'formal_pct': 0.0,
                          'consultative

## 10. Corpus + per-category aggregates

Recompute the same metric tree (a) over the entire 287-file corpus and (b) per category, so the
YAML output carries a single "corpus fingerprint" plus a row per category alongside the per-file
detail.

In [18]:
"""Aggregator over an arbitrary subset of file indices.

Strategy: every per-file record carries raw counts for every density metric.
We sum counts across the subset and recompute pct/per_sent (or sent_pct) from
totals. TTR (lexical diversity) needs the union of lemmas, so it's recomputed
from docs.

Mood no longer carries per-sentence classifications (those moved to
sentence_register). Sentence_register's per-sentence counts roll up cleanly
by simple summation.
"""

DENSITY_BLOCKS = ("register", "stance", "modality", "all_caps",
                   "caps_imperative", "justification")
MODALITY_CLASSES = ("deontic", "epistemic", "dynamic")

def _sum_counts(records, block: str, field: str) -> int:
    """Sum ``record['metrics'][block][field]`` across the records subset."""
    return sum(int(r["metrics"][block].get(field, 0) or 0) for r in records)

def _agg_block_counts(records, block: str, count_keys: list[str],
                       n_tokens_total: int, n_sents_total: int) -> dict:
    """Build {<key>_count, <key>_pct, <key>_per_sent} from per-file counts."""
    out = {}
    for k in count_keys:
        c = sum(int(r["metrics"][block].get(f"{k}_count", 0) or 0) for r in records)
        out[f"{k}_count"]    = c
        out[f"{k}_pct"]      = pct(c, n_tokens_total)
        out[f"{k}_per_sent"] = per_sent(c, n_sents_total)
    return out

def _agg_sent_pct_block(records, block: str, class_keys: list[str],
                         n_sents_total: int) -> dict:
    """Build {<key>_sent_count, <key>_sent_pct} for blocks whose denominator
    is total sentences (currently: sentence_register)."""
    out = {}
    for k in class_keys:
        c = sum(int(r["metrics"][block].get(f"{k}_sent_count", 0) or 0)
                for r in records)
        out[f"{k}_sent_count"] = c
        out[f"{k}_sent_pct"]   = round(100.0 * c / n_sents_total, 4) if n_sents_total else 0.0
    return out

def _agg_top_strings(records, block: str, field: str) -> str | None:
    """Return the most frequent value of records[*]['metrics'][block][field]."""
    c: Counter = Counter()
    for r in records:
        v = r["metrics"][block].get(field)
        if v:
            c[v] += 1
    return c.most_common(1)[0][0] if c else None

def aggregate(indices: List[int]) -> dict:
    subset = [per_file_records[i] for i in indices]
    n_tokens_total = sum(r["n_tokens"] for r in subset)
    n_sents_total  = sum(r["n_sents"] for r in subset)

    # --- Mood: lexical-density only (sums cleanly).
    marker_total = _sum_counts(subset, "mood", "marker_count")
    mood_block = {
        "marker_count":    marker_total,
        "marker_pct":      pct(marker_total, n_tokens_total),
        "marker_per_sent": per_sent(marker_total, n_sents_total),
    }

    # --- Register: numeric metrics need raw recomputation; class densities sum
    union_lemmas: List[str] = []
    sent_len_sum = sent_count = depths_sum = depths_count = 0
    formal = informal = 0
    for i in indices:
        d = docs[i]
        for t in d:
            if t.is_space:
                continue
            if not t.is_punct and not t.is_stop:
                union_lemmas.append(t.lemma_.lower())
            if t.pos_ in FORMAL_POS:
                formal += 1
            elif t.pos_ in INFORMAL_POS:
                informal += 1
        for s in d.sents:
            if s.text.strip():
                sent_len_sum += sum(1 for t in s if not t.is_space)
                sent_count += 1
        for d_ in doc_dep_depths(d):
            depths_sum += d_
            depths_count += 1
    reg_classes = list(REGISTER_MARKERS.keys())
    reg_counts = {k: _sum_counts(subset, "register", f"{k}_count") for k in reg_classes}
    register_block = {
        "ttr": (round(len(set(union_lemmas)) / len(union_lemmas), 4)
                if union_lemmas else 0.0),
        "mean_sent_len": round(sent_len_sum / sent_count, 2) if sent_count else 0.0,
        "dep_depth":     round(depths_sum / depths_count, 3) if depths_count else 0.0,
        "f_score":       (round(50 + 50 * (formal - informal) / (formal + informal), 2)
                          if (formal + informal) else 50.0),
        **{f"{k}_count":    v                                for k, v in reg_counts.items()},
        **{f"{k}_pct":      pct(v, n_tokens_total)           for k, v in reg_counts.items()},
        **{f"{k}_per_sent": per_sent(v, n_sents_total)       for k, v in reg_counts.items()},
        "dominant_register": (max(reg_counts, key=reg_counts.get)
                              if reg_counts and max(reg_counts.values()) > 0 else "none"),
    }

    # --- Stance: 5-class + 1p/2p engagement
    stance_classes = list(STANCE_MARKERS.keys()) + ["pronouns_1p", "pronouns_2p"]
    st_counts = {k: _sum_counts(subset, "stance", f"{k}_count") for k in stance_classes}
    stance_block = {
        **{f"{k}_count":    v                          for k, v in st_counts.items()},
        **{f"{k}_pct":      pct(v, n_tokens_total)     for k, v in st_counts.items()},
        **{f"{k}_per_sent": per_sent(v, n_sents_total) for k, v in st_counts.items()},
        "dominant_stance": (max((k for k in STANCE_MARKERS),
                                key=lambda k: st_counts[k])
                            if any(st_counts[k] for k in STANCE_MARKERS) else "none"),
    }

    # --- Sentence register: 6 multi-label classes + none, sent_pct denominator
    sr_block = _agg_sent_pct_block(subset, "sentence_register",
                                    SENT_REGISTER_CLASSES, n_sents_total)
    none_total = _sum_counts(subset, "sentence_register", "none_sent_count")
    sr_block["none_sent_count"] = none_total
    sr_block["none_sent_pct"]   = (round(100.0 * none_total / n_sents_total, 4)
                                    if n_sents_total else 0.0)
    nonzero = [(cls, sr_block[f"{cls}_sent_count"])
               for cls in SENT_REGISTER_CLASSES
               if sr_block[f"{cls}_sent_count"] > 0]
    sr_block["dominant"] = (max(nonzero, key=lambda kv: kv[1])[0]
                             if nonzero else "")

    # --- Modality: deontic/epistemic/dynamic + top_construction
    modality_block = _agg_block_counts(subset, "modality", list(MODALITY_CLASSES),
                                        n_tokens_total, n_sents_total)
    modality_block["top_construction"] = _agg_top_strings(subset, "modality", "top_construction")

    # --- VOCAB profile (count/pct/per_sent per key)
    vocab_block = {}
    for k in VOCAB_KEYS:
        c = sum(int(r["metrics"]["vocab"][k]["count"]) for r in subset)
        vocab_block[k] = {
            "count":    c,
            "pct":      pct(c, n_tokens_total),
            "per_sent": per_sent(c, n_sents_total),
        }

    # --- ALL CAPS: re-collect tokens for the corpus-level top list
    all_caps_counter = Counter()
    for i in indices:
        for tok in ALLCAPS_RE.findall(df.iloc[i]["raw_text"]):
            if tok not in TECH_ACRONYMS:
                all_caps_counter[tok] += 1
    total = sum(all_caps_counter.values())
    all_caps_block = {
        "count":    total,
        "distinct": len(all_caps_counter),
        "pct":      pct(total, n_tokens_total),
        "per_sent": per_sent(total, n_sents_total),
        "top":      [{"token": t, "count": c}
                     for t, c in all_caps_counter.most_common(25)],
    }

    # --- CAPS imperative
    caps_imp_counter = Counter()
    for i in indices:
        caps_imp_counter.update(CAPS_IMP_RE.findall(df.iloc[i]["raw_text"]))
    total = sum(caps_imp_counter.values())
    caps_imperative_block = {
        "count":    total,
        "pct":      pct(total, n_tokens_total),
        "per_sent": per_sent(total, n_sents_total),
        "hits":     dict(caps_imp_counter.most_common()),
    }

    # --- Justification (count + ratio = count / (marker_total + 1))
    just_total = sum(int(r["metrics"]["justification"]["count"]) for r in subset)
    justification_block = {
        "count":    just_total,
        "pct":      pct(just_total, n_tokens_total),
        "per_sent": per_sent(just_total, n_sents_total),
        "ratio":    round(just_total / (marker_total + 1), 3),
    }

    return {
        "n_tokens": n_tokens_total,
        "n_sents":  n_sents_total,
        "metrics": {
            "mood":              mood_block,
            "register":          register_block,
            "stance":            stance_block,
            "sentence_register": sr_block,
            "modality":          modality_block,
            "vocab":             vocab_block,
            "all_caps":          all_caps_block,
            "caps_imperative":   caps_imperative_block,
            "justification":     justification_block,
        },
    }

corpus_block = aggregate(list(range(len(df))))
print("corpus n_tokens:", corpus_block["n_tokens"], "n_sents:", corpus_block["n_sents"])
print("corpus modality:", {k: v for k, v in corpus_block["metrics"]["modality"].items()
                            if not k.endswith("_pct") and not k.endswith("_per_sent")})
print("corpus mood:", corpus_block["metrics"]["mood"])
print("corpus stance dominant:", corpus_block["metrics"]["stance"]["dominant_stance"])
print("corpus sentence_register sent_pcts:")
for cls in SENT_REGISTER_CLASSES:
    print(f"  {cls:>15}: {corpus_block['metrics']['sentence_register'][f'{cls}_sent_pct']:>6.2f}%  "
          f"(n={corpus_block['metrics']['sentence_register'][f'{cls}_sent_count']})")
print(f"  {'none':>15}: {corpus_block['metrics']['sentence_register']['none_sent_pct']:>6.2f}%  "
      f"(n={corpus_block['metrics']['sentence_register']['none_sent_count']})")
print(f"  dominant: {corpus_block['metrics']['sentence_register']['dominant']}")


corpus n_tokens: 129311 n_sents: 5694
corpus modality: {'deontic_count': 263, 'epistemic_count': 325, 'dynamic_count': 517, 'top_construction': 'can'}
corpus mood: {'marker_count': 1026, 'marker_pct': 0.7934, 'marker_per_sent': 0.1802}
corpus stance dominant: expository
corpus sentence_register sent_pcts:
    collaborative:   0.51%  (n=29)
       permissive:   2.21%  (n=126)
     appreciative:   0.07%  (n=4)
       imperative:  30.84%  (n=1756)
        directive:  13.91%  (n=792)
      configuring:   5.20%  (n=296)
             none:  57.94%  (n=3299)
  dominant: imperative


In [19]:
"""Per-category aggregates."""

by_category: Dict[str, dict] = {}
for cat, sub in df.groupby("category"):
    indices = sub.index.tolist()
    block = aggregate(indices)
    block["n_files"] = len(indices)
    by_category[cat] = block

for cat, b in by_category.items():
    m  = b["metrics"]["mood"]
    sr = b["metrics"]["sentence_register"]
    print(f"{cat:18s} files={b['n_files']:3d}  tokens={b['n_tokens']:6d}  "
          f"imp_sent%={sr['imperative_sent_pct']:5.1f}  "
          f"dir_sent%={sr['directive_sent_pct']:5.1f}  "
          f"cfg_sent%={sr['configuring_sent_pct']:5.1f}  "
          f"marker%={m['marker_pct']:5.2f}")


Agent prompt       files= 37  tokens= 27005  imp_sent%= 27.9  dir_sent%= 16.8  cfg_sent%=  5.9  marker%= 0.85
Data / template    files= 38  tokens= 30289  imp_sent%= 22.7  dir_sent%=  6.6  cfg_sent%=  5.2  marker%= 0.55
Skill              files= 30  tokens= 38484  imp_sent%= 33.0  dir_sent%= 11.6  cfg_sent%=  4.8  marker%= 0.65
System prompt      files= 63  tokens= 15095  imp_sent%= 39.6  dir_sent%= 20.7  cfg_sent%=  4.7  marker%= 0.93
System reminder    files= 40  tokens=  3762  imp_sent%= 44.0  dir_sent%= 28.6  cfg_sent%=  2.9  marker%= 1.78
Tool description   files= 78  tokens= 14497  imp_sent%= 37.3  dir_sent%= 23.5  cfg_sent%=  5.2  marker%= 1.19
Tool parameter     files=  1  tokens=   179  imp_sent%= 78.6  dir_sent%=  0.0  cfg_sent%= 50.0  marker%= 0.00


## 11. Write single YAML output

`prompt_linguistic_analysis.yaml` carries: metadata, all lexicons (round-tripped), corpus
aggregate, per-category aggregates, and the per-file metrics list.

In [20]:
"""Compose and write the single YAML output."""

output = {
    "metadata": {
        "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "spacy_model": f"{NLP.meta['name']}-{NLP.meta['version']}",
        "source_dir": str(CORPUS_DIR.relative_to(PROJECT_ROOT)),
        "n_files": len(df),
        "total_tokens": int(df["n_tokens"].sum()),
        "total_sents":  int(df["n_sents"].sum()),
    },
    "lexicons": {
        "VOCAB": VOCAB,
        "IMPERATIVE_MARKERS": IMPERATIVE_MARKERS,
        "CAPS_IMPERATIVE_TOKENS": CAPS_IMPERATIVE_TOKENS,
        "JUSTIFICATION_PATTERNS": JUSTIFICATION_PATTERNS,
        "TECH_ACRONYMS": sorted(TECH_ACRONYMS),
        "CATEGORY_PREFIXES": [list(p) for p in CATEGORY_PREFIXES],
        "STANCE_MARKERS": STANCE_MARKERS,
        "REGISTER_MARKERS": REGISTER_MARKERS,
        "SENTENCE_REGISTER_MARKERS": SENTENCE_REGISTER_MARKERS,
        # Modality detector parameters (no MODALITY_VERBS_REGEX — replaced by parse-tree patterns)
        "SIMPLE_MODAL_LOOKUP":       SIMPLE_MODAL_LOOKUP,
        "EPISTEMIC_ADVERBS":         sorted(EPISTEMIC_ADVERBS),
        "DEONTIC_LIGHT_VERBS":       sorted(DEONTIC_LIGHT_VERBS),
        "EPISTEMIC_LIGHT_VERBS":     sorted(EPISTEMIC_LIGHT_VERBS),
        "EPISTEMIC_AUX_AFTER_MODAL": sorted(EPISTEMIC_AUX_AFTER_MODAL),
    },
    "corpus": corpus_block,
    "by_category": by_category,
    "files": per_file_records,
}

with open(YAML_OUT, "w") as f:
    yaml.safe_dump(output, f, sort_keys=False, allow_unicode=True, width=120)

size = YAML_OUT.stat().st_size
print(f"wrote {YAML_OUT}")
print(f"      {size:,} bytes ({size/1024:.1f} KiB)")

# Round-trip integrity check
with open(YAML_OUT) as f:
    reloaded = yaml.safe_load(f)
assert set(reloaded.keys()) == {"metadata", "lexicons", "corpus", "by_category", "files"}
assert len(reloaded["files"]) == 287
assert "MODALITY_VERBS_REGEX" not in reloaded["lexicons"]
assert "SIMPLE_MODAL_LOOKUP" in reloaded["lexicons"]
assert "SENTENCE_REGISTER_MARKERS" in reloaded["lexicons"]
assert set(reloaded["lexicons"]["SENTENCE_REGISTER_MARKERS"].keys()) == {
    "collaborative", "permissive", "appreciative", "configuring"}
assert "evaluative" not in reloaded["lexicons"]["STANCE_MARKERS"]
assert "positive_evaluative" in reloaded["lexicons"]["STANCE_MARKERS"]
assert "negative_evaluative" in reloaded["lexicons"]["STANCE_MARKERS"]
sr0 = reloaded["files"][0]["metrics"]["sentence_register"]
assert "collaborative_sent_pct" in sr0
assert "imperative_sent_pct" in sr0
assert "imperative_sent_pct" not in reloaded["files"][0]["metrics"]["mood"]
print("round-trip OK — top-level keys:", list(reloaded.keys()))
print("round-trip OK — files:", len(reloaded["files"]))
print("round-trip OK — lexicons:", list(reloaded["lexicons"].keys()))
print("round-trip OK — STANCE_MARKERS keys:", list(reloaded["lexicons"]["STANCE_MARKERS"].keys()))
print("round-trip OK — SENTENCE_REGISTER_MARKERS keys:",
      list(reloaded["lexicons"]["SENTENCE_REGISTER_MARKERS"].keys()))


wrote /home/user/workspace/claude-prompts-analysis/prompt_linguistic_analysis.yaml
      1,067,222 bytes (1042.2 KiB)
round-trip OK — top-level keys: ['metadata', 'lexicons', 'corpus', 'by_category', 'files']
round-trip OK — files: 287
round-trip OK — lexicons: ['VOCAB', 'IMPERATIVE_MARKERS', 'CAPS_IMPERATIVE_TOKENS', 'JUSTIFICATION_PATTERNS', 'TECH_ACRONYMS', 'CATEGORY_PREFIXES', 'STANCE_MARKERS', 'REGISTER_MARKERS', 'SENTENCE_REGISTER_MARKERS', 'SIMPLE_MODAL_LOOKUP', 'EPISTEMIC_ADVERBS', 'DEONTIC_LIGHT_VERBS', 'EPISTEMIC_LIGHT_VERBS', 'EPISTEMIC_AUX_AFTER_MODAL']
round-trip OK — STANCE_MARKERS keys: ['directive', 'expository', 'positive_evaluative', 'negative_evaluative', 'dialogic']
round-trip OK — SENTENCE_REGISTER_MARKERS keys: ['collaborative', 'permissive', 'appreciative', 'configuring']


## 12. Visualisations

In-notebook plots only (the YAML is the persistent data output). All charts are rendered via Altair (Vega-Lite) — interactive tooltips, legend selection, and brushing where applicable.

The first batch (below) gives static-style preview charts: sentence-register profile, stance heatmap, modality grouped bars, emphasis density, and per-file outliers. The second batch (under "Interactive exploration with Altair") builds the more elaborate brushable / linked dashboards.

In [ ]:
"""Visualisations: shared theme + colour palette, then per-dimension Altair charts.

All charts in this notebook are now rendered via Altair (interactive Vega-Lite).
The cells in this section give static-style preview charts; the cells under
"Interactive exploration with Altair" further down build the brushable / linked
dashboards.
"""

%pip install --quiet altair vega_datasets

import altair as alt
alt.data_transformers.disable_max_rows()

cats = list(by_category.keys())

# Tableau 10, mapped consistently across every chart in the notebook.
TABLEAU10 = ["#4e79a7", "#f28e2c", "#e15759", "#76b7b2", "#59a14f",
             "#edc949", "#af7aa1", "#ff9da7", "#9c755f", "#bab0ab"]
CATEGORY_COLORS = {c: TABLEAU10[i % len(TABLEAU10)] for i, c in enumerate(cats)}

# Fixed 6-tone palette for sentence_register classes.
SR_CLASS_COLORS = {
    "collaborative": "#4e79a7",
    "permissive":    "#76b7b2",
    "appreciative":  "#59a14f",
    "imperative":    "#e15759",
    "directive":     "#f28e2c",
    "configuring":   "#af7aa1",
}

# Sentence-register profile per category — Altair grouped bar.
# Multi-label, six classes; near-zero `collaborative` and `appreciative` columns
# are an intentional finding: Claude Code system prompts are functional/directive,
# not interpersonal.
sr_static_long = pd.DataFrame([
    {"category": cat, "class": cls,
     "sent_pct": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_pct"],
     "sent_count": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_count"]}
    for cat in cats for cls in SENT_REGISTER_CLASSES
])

sr_static_chart = (
    alt.Chart(sr_static_long)
    .mark_bar()
    .encode(
        x=alt.X("class:N", sort=SENT_REGISTER_CLASSES, title=None,
                axis=alt.Axis(labelAngle=-30)),
        y=alt.Y("sent_pct:Q", title="% of sentences"),
        color=alt.Color("class:N",
                         scale=alt.Scale(domain=SENT_REGISTER_CLASSES,
                                         range=[SR_CLASS_COLORS[c] for c in SENT_REGISTER_CLASSES]),
                         legend=alt.Legend(title="class", orient="bottom", columns=6)),
        column=alt.Column("category:N", sort=cats, title=None,
                            header=alt.Header(labelAngle=-15, labelAlign="right")),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("class:N"),
            alt.Tooltip("sent_pct:Q",   format=".2f", title="sent %"),
            alt.Tooltip("sent_count:Q", title="sentences"),
        ],
    )
    .properties(width=130, height=200,
                title="Sentence-register profile per category — 6 classes "
                       "(multi-label; pcts may sum >100%)")
)
sr_static_chart


In [ ]:
"""Stance heatmap — Altair (per category × five stance classes + 1p/2p engagement).

The original `evaluative` row was split into `positive_evaluative` and
`negative_evaluative` to surface sentiment polarity as a first-class signal.
"""

stance_cols_static = ["directive_pct", "expository_pct",
                       "positive_evaluative_pct", "negative_evaluative_pct",
                       "dialogic_pct",
                       "pronouns_2p_pct", "pronouns_1p_pct"]

stance_long_static = pd.DataFrame([
    {"category": cat, "metric": k.replace("_pct", ""),
     "value": by_category[cat]["metrics"]["stance"][k]}
    for cat in cats for k in stance_cols_static
])

base_stance_heat = (
    alt.Chart(stance_long_static)
    .encode(
        x=alt.X("metric:N",
                sort=[k.replace("_pct", "") for k in stance_cols_static],
                title=None,
                axis=alt.Axis(labelAngle=-30)),
        y=alt.Y("category:N", sort=cats, title=None),
    )
)
heat_rect = base_stance_heat.mark_rect().encode(
    color=alt.Color("value:Q", scale=alt.Scale(scheme="magma", reverse=True),
                     title="% of file tokens"),
    tooltip=[alt.Tooltip("category:N"),
             alt.Tooltip("metric:N"),
             alt.Tooltip("value:Q", format=".3f")],
)
heat_text = base_stance_heat.mark_text(baseline="middle", fontSize=11).encode(
    text=alt.Text("value:Q", format=".2f"),
    color=alt.condition("datum.value > 1.0", alt.value("white"), alt.value("black")),
)

(heat_rect + heat_text).properties(
    width=520, height=260,
    title="Stance & engagement per category (% of file tokens) — 5-class polarity-split"
)


In [ ]:
"""Modality (deontic / epistemic / dynamic) per category — Altair grouped bars."""

mod_keys_static = ["deontic_pct", "epistemic_pct", "dynamic_pct"]
mod_long_static = pd.DataFrame([
    {"category": cat, "class": k.replace("_pct", ""),
     "value": by_category[cat]["metrics"]["modality"][k]}
    for cat in cats for k in mod_keys_static
])

mod_class_colors = {"deontic": "#e15759", "epistemic": "#4e79a7", "dynamic": "#59a14f"}

modality_chart_static = (
    alt.Chart(mod_long_static)
    .mark_bar()
    .encode(
        x=alt.X("class:N", sort=["deontic", "epistemic", "dynamic"], title=None),
        y=alt.Y("value:Q", title="% of file tokens"),
        color=alt.Color("class:N",
                         scale=alt.Scale(domain=list(mod_class_colors.keys()),
                                         range=list(mod_class_colors.values())),
                         legend=alt.Legend(title="class", orient="bottom", columns=3)),
        column=alt.Column("category:N", sort=cats, title=None,
                           header=alt.Header(labelAngle=-15, labelAlign="right")),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("class:N"),
                 alt.Tooltip("value:Q", format=".3f", title="% of tokens")],
    )
    .properties(width=110, height=220,
                title="Modality per category (% of file tokens)")
)
modality_chart_static


In [ ]:
"""Emphasis: ALL CAPS, CAPS imperative, justification ratio per category — Altair."""

emphasis_long = pd.DataFrame([
    {"category": cat, "metric": metric, "value": value}
    for cat in cats
    for metric, value in [
        ("ALL CAPS (% tokens)",
            by_category[cat]["metrics"]["all_caps"]["pct"]),
        ("CAPS imperative (% tokens)",
            by_category[cat]["metrics"]["caps_imperative"]["pct"]),
        ("Justification ratio",
            by_category[cat]["metrics"]["justification"]["ratio"]),
    ]
])

emphasis_chart = (
    alt.Chart(emphasis_long)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None),
        y=alt.Y("category:N", sort=cats, title=None),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=cats,
                                         range=[CATEGORY_COLORS[c] for c in cats]),
                         legend=None),
        column=alt.Column("metric:N", title=None,
                            sort=["ALL CAPS (% tokens)",
                                  "CAPS imperative (% tokens)",
                                  "Justification ratio"]),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("metric:N"),
                 alt.Tooltip("value:Q", format=".3f")],
    )
    .resolve_scale(x="independent")
    .properties(width=240, height=240,
                title="Emphasis density per category (independent x-scales)")
)
emphasis_chart


In [ ]:
"""Per-file outliers: highest CAPS-imperative density and lowest justification ratio."""

per_file_df = pd.DataFrame([
    {
        "path": r["path"],
        "category": r["category"],
        "n_tokens": r["n_tokens"],
        "imp_sent_pct":     r["metrics"]["sentence_register"]["imperative_sent_pct"],
        "caps_imp_pct":     r["metrics"]["caps_imperative"]["pct"],
        "all_caps_pct":     r["metrics"]["all_caps"]["pct"],
        "just_ratio":       r["metrics"]["justification"]["ratio"],
        "deontic_pct":      r["metrics"]["modality"]["deontic_pct"],
        "hard_proh_pct":    r["metrics"]["vocab"]["hard_prohibitions"]["pct"],
    }
    for r in per_file_records
])

print("--- 10 files with highest CAPS-imperative density (% of file tokens) ---")
print(per_file_df.nlargest(10, "caps_imp_pct")[["path", "category", "n_tokens", "caps_imp_pct"]].to_string(index=False))
print("\n--- 10 files with highest hard_prohibitions density (% of file tokens) ---")
print(per_file_df.nlargest(10, "hard_proh_pct")[["path", "category", "n_tokens", "hard_proh_pct"]].to_string(index=False))
print("\n--- 10 files with most explanatory tone (highest justification ratio, ≥150 tokens) ---")
big = per_file_df[per_file_df["n_tokens"] >= 150]
print(big.nlargest(10, "just_ratio")[["path", "category", "n_tokens", "just_ratio"]].to_string(index=False))


In [ ]:
"""Per-file outliers for sentence_register — Altair 4-panel.

Top 10 files by each ATTESTED class. Near-zero classes (collaborative,
appreciative) will surface their few outlier files; if a class has fewer
than 10 nonzero files, the panel renders only what exists.
"""

sr_per_file_df = pd.DataFrame([
    {
        "path": r["path"],
        "category": r["category"],
        **{f"{cls}_sent_pct": r["metrics"]["sentence_register"][f"{cls}_sent_pct"]
           for cls in SENT_REGISTER_CLASSES},
    }
    for r in per_file_records
])

panels = []
for cls in ["collaborative", "permissive", "appreciative", "configuring"]:
    col = f"{cls}_sent_pct"
    top = sr_per_file_df.nlargest(10, col).copy()
    top = top[top[col] > 0]   # drop zero-pct entries (relevant for the rare classes)
    panel = (
        alt.Chart(top)
        .mark_bar()
        .encode(
            x=alt.X(f"{col}:Q", title="% of sentences"),
            y=alt.Y("path:N", sort="-x", title=None,
                    axis=alt.Axis(labelLimit=300, labelFontSize=9)),
            color=alt.Color("category:N",
                             scale=alt.Scale(domain=cats,
                                             range=[CATEGORY_COLORS[c] for c in cats]),
                             legend=alt.Legend(title="Category", orient="bottom",
                                                columns=len(cats))),
            tooltip=[
                alt.Tooltip("path:N"),
                alt.Tooltip("category:N"),
                alt.Tooltip(f"{col}:Q", format=".2f", title=f"{cls} sent %"),
            ],
        )
        .properties(width=320, height=240,
                    title=f"Top by `{cls}_sent_pct` (n={len(top)})")
    )
    panels.append(panel)

(panels[0] | panels[1]) & (panels[2] | panels[3])


## 13. Findings

Headline patterns from this analysis (densities are **% of file tokens** unless stated otherwise; sentence-register figures are **% of sentences**).

- **The corpus is overwhelmingly directive prose, not conversation.** Across all 287 files, ~31 % of *sentences* are imperative (per `metrics.sentence_register.imperative_sent_pct`), ~14 % carry a directive marker, and only ~6 % concern configuration. The lexicon confirms: **628** hard-prohibitions vs **0** profanity hits; **1387** second-person ("you") pronouns vs **153** first-person ("we/I") — a **9:1** ratio.

- **Absence is the signal — what the corpus is NOT.** The new sentence-level pragmatic register classifier surfaces three **near-zero** classes corpus-wide:
  - `collaborative_sent_pct` ≈ **0.5 %** (29 sentences out of 5,694 — interpersonal "let's", "we should", "together").
  - `appreciative_sent_pct` ≈ **0.07 %** (just **4 sentences** out of 5,694 — gratitude, praise, thanks).
  - `permissive_sent_pct` ≈ **2.2 %** (126 sentences — "please", "you can", "you may").
  Claude Code system prompts are functional/directive, not interpersonal. Knowing what the corpus *isn't* doing is as important as knowing what it is — these classes are kept deliberately so the absence is measurable.

- **Tool descriptions are the most prohibition-heavy category.** They have the highest density of `hard_prohibitions` (~1.18 % of tokens, ≈ 1 prohibition every 85 tokens) and the highest density of imperative markers. The most extreme outlier files are the bash-sandbox tool descriptions (`bash-sandbox-no-exceptions.md`, `bash-sandbox-evidence-operation-not-permitted.md`), which run at 9 % hard-prohibitions per token — about one prohibition every 11 tokens.

- **System reminders are the most CAPS-heavy.** ALL CAPS tokens make up ~7.6 % of their tokens (≈ 4× any other category). They're terse, declarative-leaning, and emphatic.

- **Stance polarity is heavily positive.** With `evaluative` split into `positive_evaluative` and `negative_evaluative`, the corpus shows ~3–6× more positive than negative judgment markers across nearly every category. The signal is "this is the right way", not "that was the wrong way".

- **Skills and Agent prompts lead on justification ratio** (~0.41 and 0.36). These prompts pair rules with stated reasons. Tool descriptions (0.15) and System reminders (0.10) issue rules without explanations. The single file with the highest ratio is `agent-prompt-agent-creation-architect.md` at 3.0.

- **Modality (single-source spaCy detector) — what dominates per category:**
  - *Tool description* leads on deontic (0.50 % of tokens) and dynamic (0.65 %) — instructional "must / can do this" language with light epistemic content.
  - *System reminder* has the highest deontic density (0.88 %) — short emphatic obligation prompts.
  - *Skill* and *Data / template* are modality-light overall — they describe rather than command.
  - The top corpus-wide modal construction is `can` (post-audit; previously `should be`).

- **Stance fingerprint per category:**
  - *Tool description* and *System prompt* lean **directive** (highest density of `must` / `should` / `do not` / `you must`).
  - *Skill*, *Data / template*, and *Agent prompt* lean **expository** (`is a`, `means`, `for example`).
  - *System reminder* mixes directive and expository, with the highest dialogic density (`but` / `however` / `though`) — they often inject a contrastive note.

- **CAPS-imperative tokens cluster around safety-critical content.** Top emphatic tokens corpus-wide: `IMPORTANT (37)`, `NEVER (27)`, `MUST (21)`, `CRITICAL (17)`, `DO NOT (15)`. The files with the highest CAPS-imperative density are bash-sandbox descriptions, malware-analysis reminders, and chrome-browser MCP system prompts — exactly the contexts where Claude must not drift.

The full per-file numbers (one row per `.md`) and the lexicons used live in `prompt_linguistic_analysis.yaml` next to this notebook.


# Interactive exploration with Altair

The static matplotlib charts above give a quick overview. The cells below render the same data as
**Altair** (Vega-Lite) charts — interactive: hover for tooltips, brush-select to filter, click
legend entries to highlight. The charts read straight from the per-file YAML data already in
memory.

In [ ]:
"""Install + import Altair, then build a flat per-file DataFrame for charting.

The flat columns follow the rule ``<block>_<field>`` so columns from different metric
blocks (e.g. mood vs caps_imperative) don't collide. The VOCAB block is special-cased
because each VOCAB key has its own count/pct/per_sent triple.
"""
%pip install --quiet altair vega_datasets

import altair as alt
alt.data_transformers.disable_max_rows()
print("altair", alt.__version__)

# Block-aware abbreviations for flat column prefixes.
_BLOCK_PREFIX = {
    "mood":              "mood",
    "register":          "",          # numeric metrics live at top level: ttr, mean_sent_len, ...
    "stance":            "",          # already class-named: directive_pct, expository_pct, ...
    "sentence_register": "",          # already class-named: collaborative_sent_pct, ...
    "modality":          "",          # already class-named: deontic_pct, epistemic_pct, dynamic_pct
    "all_caps":          "all_caps",
    "caps_imperative":   "caps_imp",
    "justification":     "just",
}

def _flatten(rec: dict) -> dict:
    out = {k: rec.get(k) or rec.get(k, "")
           for k in ("path", "category", "name", "description",
                     "ccVersion", "agentType", "n_tokens", "n_sents")}
    for block_name, block in rec["metrics"].items():
        if block_name == "vocab":
            for key, sub in block.items():
                out[f"{key}_count"]    = sub["count"]
                out[f"{key}_pct"]      = sub["pct"]
                out[f"{key}_per_sent"] = sub["per_sent"]
            continue
        prefix = _BLOCK_PREFIX.get(block_name, block_name)
        for k, v in block.items():
            if not isinstance(v, (int, float, str, type(None))):
                continue
            col = f"{prefix}_{k}" if prefix else k
            out[col] = v
    return out

alt_df = pd.DataFrame([_flatten(r) for r in per_file_records])

# ccVersion sort key + minor bucket
def _ver_key(v: str) -> tuple:
    if not v:
        return (-1, -1, -1)
    parts = v.split(".")
    out = []
    for p in parts[:3]:
        try:    out.append(int(p))
        except ValueError: out.append(0)
    while len(out) < 3:
        out.append(0)
    return tuple(out)

alt_df["ccVersion_sort"] = alt_df["ccVersion"].apply(_ver_key)
alt_df["ccMinor"] = alt_df["ccVersion"].apply(
    lambda v: f"{v.split('.')[0]}.{v.split('.')[1]}" if v.count(".") >= 2 else (v or "unknown"))

print(alt_df.shape, "→ columns:", len(alt_df.columns))
print()
print("sentence_register columns:",
      [c for c in alt_df.columns if c.endswith("_sent_pct") or c.endswith("_sent_count")])
print("dominant column present:", "dominant" in alt_df.columns)
print()
print("ccVersion non-null:", (alt_df["ccVersion"] != "").sum())
print("ccVersion top 10:")
print(alt_df["ccVersion"].value_counts().head(10).to_string())
alt_df[["path", "name", "ccVersion", "ccMinor", "agentType"]].head()


### Linked dashboard: per-file scatter ↔ category averages

Brush a region in the scatter on the left to filter the bar charts on the right. Hover any point
for the file path and exact metric values. Click a category in the legend to highlight only that
category.

In [ ]:
"""Linked dashboard: file-level scatter + brush-filtered category aggregates.

Tooltips show ``name`` and ``description`` from the HTML-comment frontmatter — much
more readable than the raw filename."""

# Tableau 10 mapped to the categories present in the corpus (matches matplotlib chart 33).
_cat_domain = list(by_category.keys())
_cat_range  = [CATEGORY_COLORS[c] for c in _cat_domain]
cat_color = alt.Color("category:N",
                       scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                       legend=alt.Legend(title="Category", orient="bottom", columns=4))

brush = alt.selection_interval(encodings=["x", "y"])
legend_sel = alt.selection_point(fields=["category"], bind="legend")

scatter = (
    alt.Chart(alt_df)
    .mark_circle(opacity=0.7)
    .encode(
        x=alt.X("mood_marker_pct:Q",
                title="Imperative-marker density (% of file tokens)"),
        y=alt.Y("just_ratio:Q", title="Justification ratio (reasons / imperative)"),
        size=alt.Size("n_tokens:Q",
                      title="tokens",
                      scale=alt.Scale(range=[20, 600])),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.85), alt.value(0.07)),
        tooltip=[
            alt.Tooltip("name:N",        title="Name"),
            alt.Tooltip("description:N", title="Description"),
            alt.Tooltip("ccVersion:N",   title="ccVersion"),
            alt.Tooltip("path:N",        title="File"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q",    format=","),
            alt.Tooltip("imperative_sent_pct:Q",       title="imperative % of sents", format=".1f"),
            alt.Tooltip("mood_marker_pct:Q",           title="imp markers %",         format=".2f"),
            alt.Tooltip("just_ratio:Q",                title="justification ratio",   format=".2f"),
            alt.Tooltip("hard_prohibitions_pct:Q",     title="hard_proh %",           format=".2f"),
            alt.Tooltip("caps_imp_pct:Q",              title="CAPS imp %",            format=".2f"),
            alt.Tooltip("dominant_stance:N"),
            alt.Tooltip("dominant_register:N"),
            alt.Tooltip("dominant:N",                  title="dominant sentence-register"),
        ],
    )
    .add_params(brush, legend_sel)
    .properties(width=470, height=420,
                title="Per-file: imperative density vs justification ratio (brush to filter →)")
)

metrics = [
    ("hard_prohibitions_pct", "Hard prohibitions %"),
    ("caps_imp_pct",          "CAPS imperative %"),
    ("all_caps_pct",          "ALL CAPS %"),
]

linked_bars = []
for col, title in metrics:
    bar = (
        alt.Chart(alt_df)
        .mark_bar()
        .encode(
            x=alt.X(f"mean({col}):Q", title=f"{title} (of file tokens)"),
            y=alt.Y("category:N", sort="-x", title=None),
            color=cat_color,
            tooltip=[
                alt.Tooltip("category:N"),
                alt.Tooltip(f"mean({col}):Q", title=f"mean {title}", format=".3f"),
                alt.Tooltip("count():Q", title="files in selection"),
            ],
        )
        .transform_filter(brush)
        .properties(width=260, height=130, title=f"{title} (mean, in selection)")
    )
    linked_bars.append(bar)

dashboard = scatter | alt.vconcat(*linked_bars)
dashboard


### Modality per category (single-source spaCy detector)

Single-panel grouped bar showing deontic / epistemic / dynamic density per category,
in % of file tokens. Hover any bar for the underlying value; click a category name in
the legend to highlight just that category.

In [ ]:
"""Modality per category × class (single-source). Bars in % of file tokens."""

mod_long = pd.DataFrame([
    {"category": cat, "class": cls,
     "value": by_category[cat]["metrics"]["modality"][f"{cls}_pct"]}
    for cat in cats
    for cls in ("deontic", "epistemic", "dynamic")
])

cat_sel = alt.selection_point(fields=["category"], bind="legend")

mod_chart = (
    alt.Chart(mod_long)
    .mark_bar()
    .encode(
        x=alt.X("class:N", title=None,
                sort=["deontic", "epistemic", "dynamic"]),
        y=alt.Y("value:Q", title="% of file tokens"),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=_cat_domain, range=_cat_range)),
        opacity=alt.condition(cat_sel, alt.value(0.95), alt.value(0.1)),
        column=alt.Column("category:N", title=None),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("class:N"),
            alt.Tooltip("value:Q", format=".3f", title="% of file tokens"),
        ],
    )
    .add_params(cat_sel)
    .properties(width=110, height=240,
                title="Modality per category (% of file tokens)")
)
mod_chart


### Sentence-register per category (per-sentence pragmatic classifier)

Six per-sentence flags (multi-label): `collaborative` / `permissive` / `appreciative` / `imperative` / `directive` / `configuring`. Pcts can sum to >100% within a category — that's intentional.

The chart is structured to make the absence of `collaborative` and `appreciative` speech across all categories immediately visible: those bars stay near zero everywhere by design.

In [ ]:
"""Sentence-register per category × class — Altair grouped bar."""

sr_long = pd.DataFrame([
    {
        "category": cat,
        "class":    cls,
        "sent_pct": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_pct"],
        "sent_count": by_category[cat]["metrics"]["sentence_register"][f"{cls}_sent_count"],
    }
    for cat in by_category
    for cls in SENT_REGISTER_CLASSES
])

sr_class_domain = SENT_REGISTER_CLASSES
sr_class_range  = [SR_CLASS_COLORS[c] for c in sr_class_domain]

sr_chart = (
    alt.Chart(sr_long)
    .mark_bar()
    .encode(
        x=alt.X("sent_pct:Q", title="% of sentences"),
        y=alt.Y("class:N", sort=sr_class_domain, title=None),
        color=alt.Color("class:N",
                         scale=alt.Scale(domain=sr_class_domain, range=sr_class_range),
                         legend=None),
        row=alt.Row("category:N", title=None,
                     header=alt.Header(labelAngle=0, labelAlign="left")),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("class:N"),
            alt.Tooltip("sent_pct:Q",   format=".2f", title="sent %"),
            alt.Tooltip("sent_count:Q", title="sentences"),
        ],
    )
    .properties(width=480, height=140,
                title="Sentence-register per category × class (multi-label, % of sentences)")
)
sr_chart


### Stance & register heatmaps + per-file justification distribution

Two heatmaps (stance × category and register × category) plus a per-file box-strip plot of
justification ratios. Hover any cell or point for exact values.

In [ ]:
"""Stance & register heatmaps + per-file justification distribution.

Stance is now 5-class (evaluative split into positive/negative polarity).
"""

stance_long = []
stance_keys = ["directive_pct", "expository_pct",
               "positive_evaluative_pct", "negative_evaluative_pct",
               "dialogic_pct", "pronouns_2p_pct", "pronouns_1p_pct"]
register_keys = ["frozen_pct", "formal_pct", "consultative_pct", "casual_pct"]

for cat, b in by_category.items():
    s = b["metrics"]["stance"]
    for k in stance_keys:
        stance_long.append({"category": cat, "metric": k.replace("_pct", ""),
                            "value": s[k], "kind": "stance"})
    r = b["metrics"]["register"]
    for k in register_keys:
        stance_long.append({"category": cat, "metric": k.replace("_pct", ""),
                            "value": r[k], "kind": "register"})
sr_long_df = pd.DataFrame(stance_long)

heat_stance = (
    alt.Chart(sr_long_df[sr_long_df["kind"] == "stance"])
    .mark_rect()
    .encode(
        x=alt.X("metric:N",
                sort=[k.replace("_pct", "") for k in stance_keys],
                title=None,
                axis=alt.Axis(labelAngle=-30)),
        y=alt.Y("category:N", title=None),
        color=alt.Color("value:Q", scale=alt.Scale(scheme="magma", reverse=True),
                         title="% of file tokens"),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("metric:N"),
                 alt.Tooltip("value:Q", format=".3f")],
    )
    .properties(width=460, height=260,
                title="Stance × category (% of file tokens; 5-class polarity-split)")
)

heat_register = (
    alt.Chart(sr_long_df[sr_long_df["kind"] == "register"])
    .mark_rect()
    .encode(
        x=alt.X("metric:N",
                sort=[k.replace("_pct", "") for k in register_keys],
                title=None),
        y=alt.Y("category:N", title=None),
        color=alt.Color("value:Q", scale=alt.Scale(scheme="viridis"),
                         title="% of file tokens"),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("metric:N"),
                 alt.Tooltip("value:Q", format=".3f")],
    )
    .properties(width=280, height=260, title="Register × category (% of file tokens)")
)

just_box = (
    alt.Chart(alt_df)
    .mark_boxplot(extent="min-max", opacity=0.55, color="#4e79a7")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("just_ratio:Q", title="Justification ratio per file"),
    )
    .properties(width=520, height=260,
                title="Per-file justification ratio by category")
)
just_strip = (
    alt.Chart(alt_df)
    .mark_circle(size=35, opacity=0.45, color="#e15759")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("just_ratio:Q"),
        xOffset="jitter:Q",
        tooltip=[alt.Tooltip("path:N"),
                 alt.Tooltip("category:N"),
                 alt.Tooltip("n_tokens:Q"),
                 alt.Tooltip("just_ratio:Q", format=".2f")],
    )
    .transform_calculate(jitter="random()-0.5")
)

(heat_stance | heat_register) & (just_box + just_strip)


### Emphasis vocabulary: top ALL CAPS tokens, CAPS imperative tokens, and full VOCAB profile

In [ ]:
"""Top-N tokens for ALL CAPS and CAPS imperative + full VOCAB heatmap."""

top_caps = pd.DataFrame(corpus_block["metrics"]["all_caps"]["top"][:25])
top_caps_chart = (
    alt.Chart(top_caps)
    .mark_bar(color="#af7aa1")
    .encode(
        x=alt.X("count:Q", title="corpus-wide count"),
        y=alt.Y("token:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("token:N"), alt.Tooltip("count:Q")],
    )
    .properties(width=320, height=380,
                title="Top 25 ALL CAPS tokens (TECH_ACRONYMS excluded)")
)

caps_imp_data = pd.DataFrame(
    [{"token": t, "count": c} for t, c in
     corpus_block["metrics"]["caps_imperative"]["hits"].items()]
)
caps_imp_chart = (
    alt.Chart(caps_imp_data)
    .mark_bar(color="#e15759")
    .encode(
        x=alt.X("count:Q", title="corpus-wide count"),
        y=alt.Y("token:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("token:N"), alt.Tooltip("count:Q")],
    )
    .properties(width=320, height=380,
                title="CAPS imperative tokens (corpus-wide)")
)

vocab_long = []
for cat, b in by_category.items():
    for key, v in b["metrics"]["vocab"].items():
        vocab_long.append({"category": cat, "vocab_key": key, "pct": v["pct"]})
vocab_df_long = pd.DataFrame(vocab_long)

vocab_chart = (
    alt.Chart(vocab_df_long)
    .mark_rect()
    .encode(
        x=alt.X("vocab_key:N", title=None, sort=list(VOCAB_KEYS)),
        y=alt.Y("category:N", title=None),
        color=alt.Color("pct:Q", scale=alt.Scale(scheme="magma", reverse=True),
                         title="% of file tokens"),
        tooltip=[alt.Tooltip("category:N"),
                 alt.Tooltip("vocab_key:N"),
                 alt.Tooltip("pct:Q", format=".3f")],
    )
    .properties(width=720, height=260,
                title="Full VOCAB profile per category (% of file tokens)")
)

(top_caps_chart | caps_imp_chart) & vocab_chart


### Register quantitative metrics

The four register numbers — `ttr`, `mean_sent_len`, `dep_depth`, `f_score` — were computed per
file but only the four lexical register classes were plotted earlier. Two views below cover them:

- **TTR vs F-score** scatter — every file as one point, coloured by category. Files cluster
  along a single F-score band (~70-80) with TTR varying mostly by file length.
- **Sentence length & dependency depth** distributions per category (box-strip).

In [ ]:
"""Register quantitative metrics: TTR×F-score + sentence length & dep-depth distributions."""

ttr_f_chart = (
    alt.Chart(alt_df)
    .mark_circle(opacity=0.65)
    .encode(
        x=alt.X("f_score:Q", title="F-score (Heylighen & Dewaele) — higher = more formal",
                scale=alt.Scale(domain=[40, 100])),
        y=alt.Y("ttr:Q", title="Type-token ratio (lex. diversity)",
                scale=alt.Scale(domain=[0, 1])),
        size=alt.Size("n_tokens:Q", title="tokens",
                       scale=alt.Scale(range=[20, 400])),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=_cat_domain, range=_cat_range)),
        tooltip=[
            alt.Tooltip("path:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q", format=","),
            alt.Tooltip("ttr:Q", format=".3f"),
            alt.Tooltip("f_score:Q", format=".1f"),
            alt.Tooltip("mean_sent_len:Q", title="mean sent len", format=".1f"),
            alt.Tooltip("dep_depth:Q", title="dep depth", format=".2f"),
        ],
    )
    .properties(width=520, height=380, title="Per-file register: TTR × F-score")
    .interactive()
)

sent_len_box = (
    alt.Chart(alt_df)
    .mark_boxplot(extent="min-max", opacity=0.55, color="#4e79a7")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("mean_sent_len:Q", title="Mean sentence length (tokens)"),
    )
    .properties(width=240, height=300, title="Sentence length per category")
)
sent_len_strip = (
    alt.Chart(alt_df)
    .mark_circle(size=30, opacity=0.45, color="#e15759")
    .encode(
        x=alt.X("category:N", sort=cats, title=None),
        y=alt.Y("mean_sent_len:Q"),
        xOffset="jitter:Q",
        tooltip=[alt.Tooltip("path:N"),
                 alt.Tooltip("mean_sent_len:Q", format=".1f")],
    )
    .transform_calculate(jitter="random()-0.5")
)

dep_box = (
    alt.Chart(alt_df)
    .mark_boxplot(extent="min-max", opacity=0.55, color="#59a14f")
    .encode(
        x=alt.X("category:N", title=None, sort=cats),
        y=alt.Y("dep_depth:Q", title="Mean dependency depth"),
    )
    .properties(width=240, height=300, title="Dependency depth per category")
)
dep_strip = (
    alt.Chart(alt_df)
    .mark_circle(size=30, opacity=0.45, color="#af7aa1")
    .encode(
        x=alt.X("category:N", sort=cats, title=None),
        y=alt.Y("dep_depth:Q"),
        xOffset="jitter:Q",
        tooltip=[alt.Tooltip("path:N"),
                 alt.Tooltip("dep_depth:Q", format=".2f")],
    )
    .transform_calculate(jitter="random()-0.5")
)

ttr_f_chart & ((sent_len_box + sent_len_strip) | (dep_box + dep_strip))


### Cross-metric correlation + per-file directiveness ranking

Two final views:

- **Correlation matrix** across the main per-file metrics. Surfaces which dimensions move
  together (e.g. imperative density should correlate with deontic modality and prohibitions)
  and which are independent.
- **Top-25 most directive files** by a composite score: imperative-marker density + hard
  prohibitions density + CAPS imperative density (each z-scored, then summed). Hover any row
  for the full path.

In [ ]:
"""Cross-metric correlation matrix across the main per-file numeric metrics.

Updated for the new schema:
  - dropped: mood_imperative_sent_pct (now in sentence_register), evaluative_pct
  - added:   positive_evaluative_pct, negative_evaluative_pct (stance polarity split)
  - added:   the six sentence_register *_sent_pct classes
"""
import numpy as np

corr_cols = [
    # mood / lexical density
    "mood_marker_pct",
    # register quantitative
    "ttr", "mean_sent_len", "dep_depth", "f_score",
    # stance (5-class)
    "directive_pct", "expository_pct",
    "positive_evaluative_pct", "negative_evaluative_pct",
    "dialogic_pct",
    # modality
    "deontic_pct", "epistemic_pct", "dynamic_pct",
    # vocab
    "hard_prohibitions_pct", "hard_prescriptions_pct",
    "soft_prescriptions_pct", "hedging_pct", "structural_markers_pct",
    "pronouns_2p_pct", "pronouns_1p_pct",
    # loudness + reasoning
    "all_caps_pct", "caps_imp_pct",
    "just_pct", "just_ratio",
    # sentence_register (per-sentence, multi-label)
    "collaborative_sent_pct", "permissive_sent_pct", "appreciative_sent_pct",
    "imperative_sent_pct", "directive_sent_pct", "configuring_sent_pct",
]
corr = alt_df[corr_cols].corr().reset_index().melt(
    id_vars="index", var_name="other", value_name="r")
corr.columns = ["x", "y", "r"]

corr_chart = (
    alt.Chart(corr)
    .mark_rect()
    .encode(
        x=alt.X("x:N", sort=corr_cols, title=None,
                axis=alt.Axis(labelAngle=-45, labelLimit=160)),
        y=alt.Y("y:N", sort=corr_cols, title=None,
                axis=alt.Axis(labelLimit=160)),
        color=alt.Color("r:Q",
                         scale=alt.Scale(scheme="redblue", domain=[-1, 1], reverse=True),
                         title="Pearson r"),
        tooltip=[alt.Tooltip("x:N"), alt.Tooltip("y:N"),
                 alt.Tooltip("r:Q", format=".2f")],
    )
    .properties(width=620, height=620,
                title=f"Cross-metric correlation (Pearson r) — {len(corr_cols)} per-file metrics")
)
corr_chart


In [ ]:
"""Top-25 most directive files by extended composite z-score.

Updated formula adds the new sentence_register signals:
  + directive_sent_pct      (raises authority)
  + configuring_sent_pct    (raises authority)
  - collaborative_sent_pct  (softens — interpersonal)
  - permissive_sent_pct     (softens — reader-agency / "you can")
  - appreciative_sent_pct   (softens — gratitude / praise)
"""

def zscore(s):
    s = s.astype(float)
    return (s - s.mean()) / (s.std(ddof=0) or 1.0)

alt_df["directiveness"] = (
    zscore(alt_df["mood_marker_pct"])
    + zscore(alt_df["hard_prohibitions_pct"])
    + zscore(alt_df["caps_imp_pct"])
    + zscore(alt_df["directive_sent_pct"])
    + zscore(alt_df["configuring_sent_pct"])
    - zscore(alt_df["collaborative_sent_pct"])
    - zscore(alt_df["permissive_sent_pct"])
    - zscore(alt_df["appreciative_sent_pct"])
)

top25 = (alt_df.nlargest(25, "directiveness")
              [["path", "category", "n_tokens", "directiveness",
                "mood_marker_pct", "hard_prohibitions_pct",
                "caps_imp_pct",
                "directive_sent_pct", "configuring_sent_pct",
                "collaborative_sent_pct", "permissive_sent_pct",
                "appreciative_sent_pct",
                "just_ratio"]]
              .reset_index(drop=True))

print(f"composite directiveness range: "
      f"{alt_df['directiveness'].min():.2f}  ↔  {alt_df['directiveness'].max():.2f}")
display(top25.style.background_gradient(subset=["directiveness"], cmap="Reds")
                   .format({"directiveness": "{:.2f}",
                            "mood_marker_pct": "{:.2f}",
                            "hard_prohibitions_pct": "{:.2f}",
                            "caps_imp_pct": "{:.2f}",
                            "directive_sent_pct": "{:.1f}",
                            "configuring_sent_pct": "{:.1f}",
                            "collaborative_sent_pct": "{:.1f}",
                            "permissive_sent_pct": "{:.1f}",
                            "appreciative_sent_pct": "{:.1f}",
                            "just_ratio": "{:.2f}"}))

top25_chart = (
    alt.Chart(top25)
    .mark_bar()
    .encode(
        x=alt.X("directiveness:Q", title="Composite z-score (higher = more directive)"),
        y=alt.Y("path:N", sort="-x", title=None,
                axis=alt.Axis(labelLimit=320)),
        color=alt.Color("category:N",
                         scale=alt.Scale(domain=_cat_domain, range=_cat_range)),
        tooltip=[
            alt.Tooltip("path:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q", format=","),
            alt.Tooltip("directiveness:Q", format=".2f"),
            alt.Tooltip("mood_marker_pct:Q",
                         title="imperative markers %", format=".2f"),
            alt.Tooltip("hard_prohibitions_pct:Q",
                         title="hard prohibitions %", format=".2f"),
            alt.Tooltip("caps_imp_pct:Q",
                         title="CAPS imperative %", format=".2f"),
            alt.Tooltip("directive_sent_pct:Q",
                         title="directive sent %", format=".1f"),
            alt.Tooltip("configuring_sent_pct:Q",
                         title="configuring sent %", format=".1f"),
            alt.Tooltip("collaborative_sent_pct:Q",
                         title="collab sent %", format=".1f"),
            alt.Tooltip("permissive_sent_pct:Q",
                         title="permissive sent %", format=".1f"),
            alt.Tooltip("just_ratio:Q", title="justification ratio", format=".2f"),
        ],
    )
    .properties(width=560, height=560,
                title="Top 25 most directive prompts (extended z-summed composite)")
)
top25_chart


### Per-word vs per-sentence comparison

Every density metric is now reported in two units. The chart below puts them side-by-side for
the most informative metrics, so you can see how the two views differ. Categories with longer
sentences (e.g. *Tool description* at ~40 tokens/sent) show a much higher per-sentence rate
than per-word percentage, while categories with short sentences (e.g. *System reminder* at
~32 tokens/sent) sit closer together.

In [ ]:
"""Per-word vs per-sentence comparison: side-by-side bars for key metrics."""

# Six representative metrics (one per dimension), with their (pct_col, per_sent_col) pair
metric_pairs = [
    ("Imperative markers",  "mood_marker_pct",        "mood_marker_per_sent"),
    ("Hard prohibitions",   "hard_prohibitions_pct",  "hard_prohibitions_per_sent"),
    ("Directive stance",    "directive_pct",          "directive_per_sent"),
    ("Hedging",             "hedging_pct",            "hedging_per_sent"),
    ("CAPS imperative",     "caps_imp_pct",           "caps_imp_per_sent"),
    ("Justifications",      "just_pct",               "just_per_sent"),
]

rows = []
for cat in cats:
    sub = alt_df[alt_df["category"] == cat]
    for label, pct_col, ps_col in metric_pairs:
        rows.append({"category": cat, "metric": label, "unit": "% of words",
                     "value": round(sub[pct_col].mean(), 4)})
        rows.append({"category": cat, "metric": label, "unit": "per sentence",
                     "value": round(sub[ps_col].mean(), 4)})
ws_df = pd.DataFrame(rows)

unit_sel = alt.selection_point(fields=["unit"], bind="legend")

ws_chart = (
    alt.Chart(ws_df)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None),
        y=alt.Y("category:N", sort="-x", title=None),
        color=alt.Color("unit:N",
                         scale=alt.Scale(range=["#4e79a7", "#e15759"]),
                         legend=alt.Legend(title="Unit", orient="bottom")),
        opacity=alt.condition(unit_sel, alt.value(0.85), alt.value(0.15)),
        row=alt.Row("metric:N", title=None,
                     sort=[label for label, _, _ in metric_pairs],
                     header=alt.Header(labelAngle=0, labelAlign="left")),
        column=alt.Column("unit:N", title=None,
                           sort=["% of words", "per sentence"]),
        tooltip=[
            alt.Tooltip("category:N"),
            alt.Tooltip("metric:N"),
            alt.Tooltip("unit:N"),
            alt.Tooltip("value:Q", format=".3f"),
        ],
    )
    .add_params(unit_sel)
    .properties(width=180, height=120,
                title="Six metrics, two unit views (per-word % vs per-sentence rate)")
    .resolve_scale(x="independent")
)
ws_chart


### ccVersion timeline

Each prompt's `ccVersion` (from the HTML-comment frontmatter) records the Claude Code
release the prompt was last touched in. The chart below puts every file at its
`ccVersion` on the x-axis and a directiveness signal on the y-axis. Hover any point for
the prompt's full `name`, `description`, and version. Brush horizontally to focus on a
release window; click a category in the legend to highlight just that category.

In [ ]:
"""ccVersion timeline + minor-version histogram, with name/description tooltips.

Directiveness composite uses the extended formula from cell 58 (mood markers +
hard prohibitions + CAPS imperatives + directive_sent_pct + configuring_sent_pct
- collaborative_sent_pct - permissive_sent_pct - appreciative_sent_pct).
"""

# Sort ccVersion values numerically (treating "2.1.53" as a tuple)
version_order = (
    alt_df[alt_df["ccVersion"] != ""]
    .drop_duplicates("ccVersion")
    .sort_values("ccVersion_sort")
    ["ccVersion"]
    .tolist()
)

minor_counts = (
    alt_df[alt_df["ccMinor"] != "unknown"]
    .groupby(["ccMinor", "category"])
    .size()
    .reset_index(name="count")
)

def _minor_key(m):
    parts = m.split(".")
    try:
        return (int(parts[0]), int(parts[1]) if len(parts) > 1 else 0)
    except (ValueError, IndexError):
        return (-1, -1)
minor_order = sorted(minor_counts["ccMinor"].unique(), key=_minor_key)

cat_color = alt.Color("category:N",
                       scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                       legend=alt.Legend(title="Category", orient="bottom", columns=4))
legend_sel = alt.selection_point(fields=["category"], bind="legend")

# Use the precomputed `directiveness` from cell 58 — same formula.
df_with_ver = alt_df[alt_df["ccVersion"] != ""].copy()

timeline = (
    alt.Chart(df_with_ver)
    .mark_circle(opacity=0.65)
    .encode(
        x=alt.X("ccVersion:N", sort=version_order, title="ccVersion (oldest → newest)",
                axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
        y=alt.Y("directiveness:Q", title="Composite directiveness (z-score, extended)"),
        size=alt.Size("n_tokens:Q", scale=alt.Scale(range=[20, 400]), legend=None),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.8), alt.value(0.07)),
        xOffset="jitter:Q",
        tooltip=[
            alt.Tooltip("name:N",        title="Name"),
            alt.Tooltip("description:N", title="Description"),
            alt.Tooltip("ccVersion:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q",    format=","),
            alt.Tooltip("directiveness:Q", format=".2f"),
            alt.Tooltip("path:N"),
        ],
    )
    .transform_calculate(jitter="random()-0.5")
    .add_params(legend_sel)
    .properties(width=820, height=320,
                title="Per-file directiveness over ccVersion (jittered, hover for prompt name)")
)

hist = (
    alt.Chart(minor_counts)
    .mark_bar()
    .encode(
        x=alt.X("ccMinor:N", sort=minor_order,
                title="ccVersion minor (major.minor)"),
        y=alt.Y("count:Q", title="files"),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.95), alt.value(0.1)),
        tooltip=[
            alt.Tooltip("ccMinor:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("count:Q", title="files"),
        ],
    )
    .properties(width=820, height=180,
                title="Files per Claude Code minor version, stacked by category")
)

timeline & hist


### Loudness & imperative density across ccVersion

Four small multiples plotting the per-`ccVersion` mean of:

- **Loudness** — ALL CAPS density and CAPS imperative density (both % of file tokens).
- **Imperatives** — imperative-marker density (% of file tokens) and the share of sentences classified as imperative (% of sentences).

Each panel uses an independent y-axis so the absolute scales of the four metrics don't squash each other. Versions are ordered oldest → newest along x.

In [ ]:
"""Loudness & imperative-density trends across ccVersion."""

df_ver = alt_df[alt_df["ccVersion"] != ""].copy()

trend_specs = [
    ("all_caps_pct",        "Loudness — ALL CAPS density",                         "% of file tokens", "#e15759"),
    ("caps_imp_pct",        "Loudness — CAPS imperative density",                  "% of file tokens", "#af7aa1"),
    ("mood_marker_pct",     "Imperatives — imperative-marker density (per word)",  "% of file tokens", "#4e79a7"),
    ("imperative_sent_pct", "Imperatives — imperative sentences (per sentence)",   "% of sentences",   "#f28e2c"),
]

frames = []
for col, label, unit, _ in trend_specs:
    g = (
        df_ver.groupby(["ccVersion", "ccVersion_sort"])[col]
        .mean()
        .reset_index()
        .rename(columns={col: "value"})
    )
    g["metric"] = label
    g["unit"] = unit
    frames.append(g)
trend_df = pd.concat(frames, ignore_index=True)


def _trend_panel(label, unit, color, show_x_title):
    return (
        alt.Chart(trend_df[trend_df["metric"] == label])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=60),
                   strokeWidth=2.5, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion (oldest → newest)" if show_x_title else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
            y=alt.Y("value:Q", title=unit),
            tooltip=[
                alt.Tooltip("ccVersion:N"),
                alt.Tooltip("value:Q", format=".3f", title="mean"),
                alt.Tooltip("unit:N"),
            ],
        )
        .properties(width=820, height=150, title=label)
    )


panels = [
    _trend_panel(label, unit, color, show_x_title=(i == len(trend_specs) - 1))
    for i, (_, label, unit, color) in enumerate(trend_specs)
]

alt.vconcat(*panels).resolve_scale(y="independent")


### Sentence-register classes across ccVersion

Six small multiples plotting the per-`ccVersion` mean of each sentence-register class. Independent y-axes so the near-zero `collaborative` and `appreciative` panels still show their (small) trends without being squashed under the imperative scale.

**Rationale for keeping near-zero classes visible**: knowing what's absent across ccVersion is part of the picture. If the corpus ever starts using interpersonal or appreciative speech, the line will lift off zero — that lift is the signal worth catching.

In [ ]:
"""Sentence-register classes across ccVersion — 6-panel small-multiples."""

sr_trend_specs = [
    ("collaborative", "Collaborative — interpersonal 1p-plural", "#4e79a7"),
    ("permissive",    "Permissive — soft-directive permission",  "#76b7b2"),
    ("appreciative",  "Appreciative — gratitude / praise",       "#59a14f"),
    ("imperative",    "Imperative — grammatical mood",           "#e15759"),
    ("directive",     "Directive — must / should / never markers", "#f28e2c"),
    ("configuring",   "Configuring — config / parameter speech", "#af7aa1"),
]

sr_frames = []
for cls, _label, _color in sr_trend_specs:
    col = f"{cls}_sent_pct"
    g = (
        df_with_ver.groupby(["ccVersion", "ccVersion_sort"])[col]
        .mean()
        .reset_index()
        .rename(columns={col: "value"})
    )
    g["class"] = cls
    sr_frames.append(g)
sr_trend_df = pd.concat(sr_frames, ignore_index=True)


def _sr_panel(cls, label, color, show_x_title):
    return (
        alt.Chart(sr_trend_df[sr_trend_df["class"] == cls])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=60),
                   strokeWidth=2.5, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion (oldest → newest)" if show_x_title else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
            y=alt.Y("value:Q", title="% of sentences"),
            tooltip=[
                alt.Tooltip("ccVersion:N"),
                alt.Tooltip("value:Q", format=".3f", title="mean sent_pct"),
                alt.Tooltip("class:N"),
            ],
        )
        .properties(width=820, height=130, title=label)
    )


sr_panels = [
    _sr_panel(cls, label, color, show_x_title=(i == len(sr_trend_specs) - 1))
    for i, (cls, label, color) in enumerate(sr_trend_specs)
]

alt.vconcat(*sr_panels).resolve_scale(y="independent")
